<a href="https://colab.research.google.com/github/akshayagg18/ureca_attempt1/blob/main/Ureca_cfc_try1_continued.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Phase 1: CADDreamer Replication on Google Colab (T4 GPU)
# Paper: https://arxiv.org/abs/2502.20732
# Repo: https://github.com/lidan233/CADDreamer
# ============================================================
# Step 0: Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [6]:
import os, subprocess, sys
from pathlib import Path

env = os.environ.copy()
env["TCNN_CUDA_ARCHITECTURES"] = "75"
env["MAX_JOBS"] = "2"
log = Path("/content/tcnn_build.log")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja"], check=True)

with log.open("w") as f:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir",
         "git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch"],
        env=env, stdout=f, stderr=subprocess.STDOUT
    )

print("TCNN_INSTALL_RETURN_CODE", result.returncode)
print("\n".join(log.read_text(errors="replace").splitlines()[-30:]))

if result.returncode == 0:
    import tinycudann
    print("TINYCUDANN_IMPORT_PASS")
else:
    raise RuntimeError("tinycudann build failed; inspect /content/tcnn_build.log")


TCNN_INSTALL_RETURN_CODE 0
  Cloning https://github.com/NVlabs/tiny-cuda-nn/ to /tmp/pip-req-build-gm5ip9xk
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/tiny-cuda-nn/ /tmp/pip-req-build-gm5ip9xk
  Resolved https://github.com/NVlabs/tiny-cuda-nn/ to commit 749dd70c5afc5a9dadb85e5652ed65d55e0ba187
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for tinycudann: filename=tinycudann-2.0-cp312-cp312-linux_x86_64.whl size=16639591 sha256=41cbcc8a33b270e19db5cb1e15c34798741c99fc63cd7de3dc67528e982ee4a6
  Stored in directory: /tmp/pip-ephem-wheel-cache-ved8brrc/wheels/2f/e8/5a/6f5fba4370cba8f29cff0bb004adb2bfbe0a148555d848019a
Successfully built tinycudann
TINYCUDANN_IMPORT_PASS


import subprocess, sys, os

# First check if files exist (they should persist from before kernel restart)
result = subprocess.run(['ls', '/content/CADDreamer/mvdiffusion/models/'],
                       capture_output=True, text=True)
print("FILES:", result.stdout)
print("STDERR:", result.stderr)

# Check GPU
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("CHECK_DONE")

In [ ]:
%%bash
# Codex Step 24: Force the exact compatibility pins without resolver backtracking
set +e

echo "=== Force compatible versions ==="
python3 -m pip install -q --no-deps --force-reinstall --no-cache-dir \
  "huggingface_hub==0.24.7" \
  "tokenizers==0.19.1" \
  "transformers==4.44.0" \
  "accelerate==0.33.0" \
  "diffusers==0.29.2"

python3 -m pip install -q --no-deps --force-reinstall --no-cache-dir "numpy==2.4.2"

echo "=== Verify versions in a fresh process ==="
python3 - <<'PY'
import torch, diffusers, transformers, huggingface_hub, numpy, PIL
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("diffusers", diffusers.__version__)
print("transformers", transformers.__version__)
print("hub", huggingface_hub.__version__)
print("numpy", numpy.__version__)
print("PIL", PIL.__version__)
PY

echo "=== Fresh 5-module import test ==="
cd /content/CADDreamer
python3 /tmp/caddreamer_import_test.py 2>&1
%%bash
# Codex Step 23: Resolve the three exact remaining import blockers
set +e

echo "=== Pin a mutually compatible core stack ==="
python3 -m pip install -q --upgrade --force-reinstall --no-cache-dir \
  "diffusers==0.29.2" \
  "transformers==4.44.0" \
  "huggingface_hub==0.24.7" \
  "tokenizers==0.19.1" \
  "accelerate==0.33.0" \
  "numpy==2.4.2"

cd /content/CADDreamer

echo "=== Patch maybe_allow_in_graph import ==="
python3 - <<'PY'
from pathlib import Path
p = Path("/content/CADDreamer/mvdiffusion/models/transformer_mv2d.py")
s = p.read_text()
old = "from diffusers.utils import BaseOutput, deprecate, maybe_allow_in_graph"
new = """from diffusers.utils import BaseOutput, deprecate
try:
    from diffusers.utils.torch_utils import maybe_allow_in_graph
except ImportError:
    def maybe_allow_in_graph(fn):
        return fn"""
if old in s:
    s = s.replace(old, new)
    p.write_text(s)
    print("patched transformer_mv2d.py")
else:
    print("transformer_mv2d.py already patched or import changed")
PY

echo "=== Verify pinned versions ==="
python3 - <<'PY'
import torch, diffusers, transformers, huggingface_hub, numpy, PIL
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("diffusers", diffusers.__version__)
print("transformers", transformers.__version__)
print("hub", huggingface_hub.__version__)
print("numpy", numpy.__version__)
print("PIL", PIL.__version__)
PY

echo "=== Fresh 5-module import test ==="
python3 /tmp/caddreamer_import_test.py 2>&1


In [ ]:
%%bash
# Codex: clean CADDreamer T4 bootstrap + public Wonder3D checkpoint
set +e

cd /content
if [ ! -d CADDreamer ]; then
  git clone -q https://github.com/lidan233/CADDreamer.git
fi

echo "=== Install supporting dependencies ==="
python3 -m pip install -q --no-cache-dir \
  einops omegaconf rembg onnxruntime segment-anything trimesh meshio open3d \
  scikit-image opencv-python dill cupy-cuda12x

echo "=== Force the verified compatibility core ==="
python3 -m pip install -q --no-deps --force-reinstall --no-cache-dir \
  "huggingface_hub==0.24.7" "tokenizers==0.19.1" "transformers==4.44.0" \
  "accelerate==0.33.0" "diffusers==0.29.2" "numpy==2.4.2" "pillow==9.5.0"

cat > /tmp/patch_caddreamer.py <<'PY'
from pathlib import Path
import re
root = Path("/content/CADDreamer")

# UNet condition model
p = root / "mvdiffusion/models/unet_mv2d_condition.py"
s = p.read_text()
s = s.replace("from diffusers.models.modeling_utils import ModelMixin, load_state_dict, _load_state_dict_into_model",
              "from diffusers.models.modeling_utils import ModelMixin, load_state_dict")
s = s.replace("from diffusers.models.unet_2d_blocks import",
              "from diffusers.models.unets.unet_2d_blocks import")
s = s.replace("    DIFFUSERS_CACHE,\n", "").replace("    HF_HUB_OFFLINE,\n", "")
shim = '''\ntry:
    from diffusers.utils import DIFFUSERS_CACHE
except Exception:
    DIFFUSERS_CACHE = str(Path.home() / ".cache/huggingface/diffusers")
try:
    from diffusers.utils import HF_HUB_OFFLINE
except Exception:
    HF_HUB_OFFLINE = False
try:
    from diffusers.models.modeling_utils import _load_state_dict_into_model
except Exception:
    def _load_state_dict_into_model(model_to_load, state_dict):
        model_to_load.load_state_dict(state_dict, strict=False)
        return []
'''
if "DIFFUSERS_CACHE = str(Path.home()" not in s:
    s = "from pathlib import Path\n" + shim + s
p.write_text(s)

# MV blocks
p = root / "mvdiffusion/models/unet_mv2d_blocks.py"
s = p.read_text()
s = s.replace("from diffusers.models.attention import AdaGroupNorm",
              "from diffusers.models.normalization import AdaGroupNorm")
s = s.replace("from diffusers.models.dual_transformer_2d import",
              "from diffusers.models.transformers.dual_transformer_2d import")
s = s.replace("from diffusers.models.unet_2d_blocks import",
              "from diffusers.models.unets.unet_2d_blocks import")
p.write_text(s)

# Transformer decorator
p = root / "mvdiffusion/models/transformer_mv2d.py"
s = p.read_text()
s = s.replace(
    "from diffusers.utils import BaseOutput, deprecate, maybe_allow_in_graph",
    """from diffusers.utils import BaseOutput, deprecate
try:
    from diffusers.utils.torch_utils import maybe_allow_in_graph
except ImportError:
    def maybe_allow_in_graph(fn):
        return fn"""
)
p.write_text(s)

# Pipeline
p = root / "mvdiffusion/pipelines/pipeline_mvdiffusion_image.py"
s = p.read_text()
s = s.replace("from diffusers.utils import deprecate, logging, randn_tensor",
              "from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor")
s = s.replace("from diffusers.utils import deprecate, logging as dlogging, randn_tensor",
              "from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor")
s = s.replace("dlogging.get_logger", "logging.get_logger")
p.write_text(s)

# Autoencoder
p = root / "automl/AutoencoderKL.py"
s = p.read_text()
s = s.replace("from diffusers.loaders import FromOriginalVAEMixin",
              "try:\n    from diffusers.loaders import FromOriginalVAEMixin\nexcept Exception:\n    FromOriginalVAEMixin = object")
s = s.replace("from diffusers.utils import BaseOutput, apply_forward_hook",
              "from diffusers.utils import BaseOutput\ntry:\n    from diffusers.utils.accelerate_utils import apply_forward_hook\nexcept Exception:\n    apply_forward_hook = lambda fn: fn")
s = s.replace("from diffusers.models.vae import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder",
              "from diffusers.models.autoencoders.autoencoder_kl import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder")
s = s.replace("from diffusers.utils import BaseOutput, is_torch_version, randn_tensor",
              "from diffusers.utils import BaseOutput, is_torch_version\nfrom diffusers.utils.torch_utils import randn_tensor")
s = s.replace("from diffusers.models.unet_2d_blocks import",
              "from diffusers.models.unets.unet_2d_blocks import")
p.write_text(s)
print("CADDreamer compatibility patches applied")
PY

python3 /tmp/patch_caddreamer.py

cat > /tmp/test_caddreamer.py <<'PY'
import os, sys
sys.path.insert(0, "/content/CADDreamer")
os.chdir("/content/CADDreamer")
tests = [
 ("UNetMV2DConditionModel","mvdiffusion.models.unet_mv2d_condition","UNetMV2DConditionModel"),
 ("SingleImageDataset","mvdiffusion.data.single_image_dataset","SingleImageDataset"),
 ("MVDiffusionImagePipeline","mvdiffusion.pipelines.pipeline_mvdiffusion_image","MVDiffusionImagePipeline"),
 ("AutoencoderKL","automl.AutoencoderKL","AutoencoderKL"),
 ("save_cache","utils.util","save_cache"),
]
n=0
for label,mod,attr in tests:
    try:
        m=__import__(mod,fromlist=[attr]); getattr(m,attr)
        print("[PASS]",label); n+=1
    except Exception as e:
        print("[FAIL]",label,type(e).__name__,str(e)[:300])
print(f"Score: {n}/5")
raise SystemExit(0 if n==5 else 1)
PY

echo "=== Verify imports ==="
python3 /tmp/test_caddreamer.py || exit 1

echo "=== Download public Wonder3D base checkpoint ==="
python3 - <<'PY'
from huggingface_hub import snapshot_download
path = snapshot_download(
    repo_id="flamehaze1115/wonder3d-v1.0",
    local_dir="/content/CADDreamer/ckpts/wonder3d-v1.0",
    local_dir_use_symlinks=False,
    resume_download=True,
)
print("Downloaded to", path)
PY

echo "=== Base checkpoint ready ==="
du -sh /content/CADDreamer/ckpts/wonder3d-v1.0
find /content/CADDreamer/ckpts/wonder3d-v1.0 -maxdepth 2 -type f | sort


In [ ]:
# Codex: smoke-load the public Wonder3D pipeline and inspect bundled CADDreamer outputs
import os, sys, json, traceback
from pathlib import Path

import torch
from PIL import Image
from IPython.display import display
import trimesh

ROOT = Path('/content/CADDreamer')
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print('=== Environment ===')
print('cwd:', ROOT)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

print('\n=== Key path check ===')
check_paths = [
    ROOT / 'ckpts/wonder3d-v1.0/model_index.json',
    ROOT / 'cached_output/cropsize-256-cfg1.0/0_0deepcad/all_colors.png',
    ROOT / 'cached_output/cropsize-256-cfg1.0/0_0deepcad/all_normal.png',
    ROOT / 'cached_output/cropsize-256-cfg1.0/0_0deepcad/False_mm.obj',
    ROOT / 'cached_output/cropsize-256-cfg1.0/0_0deepcad/result.step',
]
for p in check_paths:
    print('[OK]' if p.exists() else '[MISSING]', p)

print('\n=== Wonder3D pipeline smoke-load ===')
try:
    from mvdiffusion.pipelines.pipeline_mvdiffusion_image import MVDiffusionImagePipeline
    print('import ok:', MVDiffusionImagePipeline.__name__)
    pipe = MVDiffusionImagePipeline.from_pretrained(
        str(ROOT / 'ckpts/wonder3d-v1.0'),
        torch_dtype=torch.float16,
        local_files_only=True,
    )
    print('PIPELINE LOAD PASS:', type(pipe).__name__)
    print('component keys:', sorted(pipe.components.keys()))
    if torch.cuda.is_available():
        pipe = pipe.to('cuda')
        print('moved pipeline to cuda')
except Exception:
    traceback.print_exc()

print('\n=== Bundled CADDreamer output inspection ===')
scene_dir = ROOT / 'cached_output/cropsize-256-cfg1.0/0_0deepcad'
if scene_dir.exists():
    for name in ['all_colors.png', 'all_normal.png']:
        img_path = scene_dir / name
        img = Image.open(img_path)
        print(name, 'size=', img.size, 'mode=', img.mode)
        display(img)

    mesh_path = scene_dir / 'False_mm.obj'
    mesh = trimesh.load(mesh_path, force='mesh')
    print('mesh vertices:', len(mesh.vertices))
    print('mesh faces:', len(mesh.faces))
    print('mesh watertight:', bool(mesh.is_watertight))
    print('step bytes:', (scene_dir / 'result.step').stat().st_size)
else:
    print('scene_dir missing:', scene_dir)

print('\n=== Remaining blockers for full Phase-1 replication ===')
required_external = [
    ROOT / 'finetuning_vae_normal/outputs/finetune_vae/vae_16.pth',
    ROOT / 'finetuning_vae_normal/outputs/finetune_vae/mlp_16.pth',
    ROOT / 'finetuning_vae_normal/outputs/finetuning_vae_normal/vae_normal_2.pth',
    ROOT / 'our_inputs/test_real_images/testnormal_0_0.png',
]
status = {str(p): p.exists() for p in required_external}
print(json.dumps(status, indent=2))

In [ ]:
%%bash
# Step 2 (Rerun): Install all dependencies
pip install -q diffusers transformers accelerate einops omegaconf rembg open3d trimesh
pip install -q "diffusers>=0.27.0" "transformers>=4.35.0"
pip install -q xformers --index-url https://download.pytorch.org/whl/cu118
echo "=== Installation complete ==="
python -c "import torch; print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"

In [ ]:
import os, sys
from pathlib import Path
print('PROBE_START')
print('python', sys.version.split()[0])
try:
    import torch
    print('torch', torch.__version__)
    print('cuda_available', torch.cuda.is_available())
    print('gpu_name', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as e:
    print('torch_fail', type(e).__name__, e)
root = Path('/content/CADDreamer')
print('repo_exists', root.exists())
print('ckpt_exists', (root / 'ckpts/wonder3d-v1.0/model_index.json').exists())
for mod in ['trimesh','diffusers','transformers','segment_anything']:
    try:
        m = __import__(mod)
        print(mod, 'OK', getattr(m, '__version__', 'no_version'))
    except Exception as e:
        print(mod, 'FAIL', type(e).__name__, e)
print('PROBE_DONE')%%bash
set -u
cd /content
if [ ! -d CADDreamer/.git ]; then
  rm -rf /content/CADDreamer
  git clone --depth 1 https://github.com/lidan233/CADDreamer.git
fi
cd /content/CADDreamer

python3 - <<'PY'
import sys, subprocess
print('=== GPU / Python environment ===')
try:
    import torch
    print('python:', sys.version.split()[0])
    print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as e:
    print('torch import issue:', e)

pkgs = [
    'huggingface_hub==0.24.7',
    'tokenizers==0.19.1',
    'transformers==4.44.0',
    'accelerate==0.33.0',
    'diffusers==0.29.2',
    'numpy==2.4.2',
    'pillow==9.5.0',
    'einops', 'omegaconf', 'rembg', 'onnxruntime', 'segment-anything',
    'trimesh', 'meshio', 'open3d', 'scikit-image', 'opencv-python', 'dill',
    'cupy-cuda12x'
]
print('=== Install/repair Python packages ===')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
PY

python3 - <<'PY'
from pathlib import Path
import re
ROOT = Path('/content/CADDreamer')

def write_if_changed(path, text):
    old = Path(path).read_text(encoding='utf-8')
    if old != text:
        Path(path).write_text(text, encoding='utf-8')
        print('patched:', path)
    else:
        print('ok:', path)

p = ROOT / 'mvdiffusion/models/unet_mv2d_condition.py'
t = p.read_text(encoding='utf-8')
t = t.replace('from diffusers.models.unet_2d_blocks import (', 'from diffusers.models.unets.unet_2d_blocks import (')
t = t.replace('from diffusers.models.dual_transformer_2d import DualTransformer2DModel', 'from diffusers.models.transformers.dual_transformer_2d import DualTransformer2DModel')
if 'from diffusers.utils.torch_utils import maybe_allow_in_graph' not in t:
    t = re.sub(r'from diffusers\.utils import \((.*?)maybe_allow_in_graph,?(.*?)\)', 'from diffusers.utils import (\\1\\2)\ntry:\n    from diffusers.utils.torch_utils import maybe_allow_in_graph\nexcept Exception:\n    def maybe_allow_in_graph(cls):\n        return cls', t, flags=re.S)
write_if_changed(p, t)

p = ROOT / 'mvdiffusion/models/unet_mv2d_blocks.py'
t = p.read_text(encoding='utf-8')
t = t.replace('from diffusers.models.unet_2d_blocks import (', 'from diffusers.models.unets.unet_2d_blocks import (')
t = t.replace('from diffusers.models.dual_transformer_2d import DualTransformer2DModel', 'from diffusers.models.transformers.dual_transformer_2d import DualTransformer2DModel')
if 'from diffusers.models.normalization import AdaGroupNorm' not in t:
    t = t.replace('from diffusers.models.attention import AdaGroupNorm', 'try:\n    from diffusers.models.normalization import AdaGroupNorm\nexcept Exception:\n    from diffusers.models.attention import AdaGroupNorm')
write_if_changed(p, t)

p = ROOT / 'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py'
t = p.read_text(encoding='utf-8')
if 'from diffusers.utils.torch_utils import randn_tensor' not in t:
    t = re.sub(r'from diffusers\.utils import \((.*?)randn_tensor,?(.*?)\)', 'from diffusers.utils import (\\1\\2)\nfrom diffusers.utils.torch_utils import randn_tensor', t, flags=re.S)
    t = t.replace('from diffusers.utils import randn_tensor', 'from diffusers.utils.torch_utils import randn_tensor')
write_if_changed(p, t)

p = ROOT / 'automl/AutoencoderKL.py'
t = p.read_text(encoding='utf-8')
t = t.replace('from diffusers.models.vae import DecoderOutput, Decoder, DiagonalGaussianDistribution, Encoder', 'from diffusers.models.autoencoders.vae import DecoderOutput, Decoder, DiagonalGaussianDistribution, Encoder')
t = t.replace('from diffusers.models.vae import DecoderOutput, DiagonalGaussianDistribution, Encoder, Decoder', 'from diffusers.models.autoencoders.vae import DecoderOutput, DiagonalGaussianDistribution, Encoder, Decoder')
t = t.replace('from diffusers.utils import BaseOutput, randn_tensor', 'from diffusers.utils import BaseOutput\nfrom diffusers.utils.torch_utils import randn_tensor')
t = t.replace('from diffusers.models.modeling_utils import ModelMixin, apply_forward_hook', 'from diffusers.models.modeling_utils import ModelMixin\nfrom diffusers.utils.accelerate_utils import apply_forward_hook')
if 'from diffusers.loaders import FromOriginalVAEMixin' in t and 'single_file_model' not in t:
    t = t.replace('from diffusers.loaders import FromOriginalVAEMixin', 'try:\n    from diffusers.loaders import FromOriginalVAEMixin\nexcept Exception:\n    from diffusers.loaders.single_file_model import FromOriginalModelMixin as FromOriginalVAEMixin')
write_if_changed(p, t)
PY

python3 - <<'PY'
print('=== Fresh 5-module import test ===')
mods = {
    'UNetMV2DConditionModel': ('mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    'SingleImageDataset': ('mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    'MVDiffusionImagePipeline': ('mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    'AutoencoderKL': ('automl.AutoencoderKL', 'AutoencoderKL'),
    'save_cache': ('dataset.save_cache', 'save_cache'),
}
score = 0
for name, (mod, attr) in mods.items():
    try:
        m = __import__(mod, fromlist=[attr])
        getattr(m, attr)
        print(f'[PASS] {name}')
        score += 1
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'[FAIL] {name}: {type(e).__name__}: {e}')
print(f'Score: {score}/{len(mods)}')
PY

python3 - <<'PY'
from pathlib import Path
from huggingface_hub import snapshot_download
root = Path('/content/CADDreamer/ckpts/wonder3d-v1.0')
if not (root / 'model_index.json').exists():
    print('=== Download public Wonder3D checkpoint ===')
    out = snapshot_download(
        repo_id='flamehaze1115/wonder3d-v1.0',
        local_dir=str(root),
        local_dir_use_symlinks=False,
        resume_download=True,
    )
    print('downloaded to', out)
else:
    print('=== Wonder3D checkpoint already present ===')
    print(root)
PY%%bash
set -u
cd /content
if [ ! -d CADDreamer/.git ]; then
  rm -rf /content/CADDreamer
  git clone --depth 1 https://github.com/lidan233/CADDreamer.git
fi
cd /content/CADDreamer

python3 - <<'PY'
import sys, subprocess
print('=== GPU / Python environment ===')
try:
    import torch
    print('python:', sys.version.split()[0])
    print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as e:
    print('torch import issue:', e)

pkgs = [
    'huggingface_hub==0.24.7',
    'tokenizers==0.19.1',
    'transformers==4.44.0',
    'accelerate==0.33.0',
    'diffusers==0.29.2',
    'numpy==2.4.2',
    'pillow==9.5.0',
    'einops', 'omegaconf', 'rembg', 'onnxruntime', 'segment-anything',
    'trimesh', 'meshio', 'open3d', 'scikit-image', 'opencv-python', 'dill',
    'cupy-cuda12x'
]
print('=== Install/repair Python packages ===')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
PY

python3 - <<'PY'
from pathlib import Path
import re
ROOT = Path('/content/CADDreamer')

def write_if_changed(path, text):
    old = Path(path).read_text(encoding='utf-8')
    if old != text:
        Path(path).write_text(text, encoding='utf-8')
        print('patched:', path)
    else:
        print('ok:', path)

p = ROOT / 'mvdiffusion/models/unet_mv2d_condition.py'
t = p.read_text(encoding='utf-8')
t = t.replace('from diffusers.models.unet_2d_blocks import (', 'from diffusers.models.unets.unet_2d_blocks import (')
t = t.replace('from diffusers.models.dual_transformer_2d import DualTransformer2DModel', 'from diffusers.models.transformers.dual_transformer_2d import DualTransformer2DModel')
if 'from diffusers.utils.torch_utils import maybe_allow_in_graph' not in t:
    t = re.sub(r'from diffusers\.utils import \((.*?)maybe_allow_in_graph,?(.*?)\)', 'from diffusers.utils import (\\1\\2)\ntry:\n    from diffusers.utils.torch_utils import maybe_allow_in_graph\nexcept Exception:\n    def maybe_allow_in_graph(cls):\n        return cls', t, flags=re.S)
write_if_changed(p, t)

p = ROOT / 'mvdiffusion/models/unet_mv2d_blocks.py'
t = p.read_text(encoding='utf-8')
t = t.replace('from diffusers.models.unet_2d_blocks import (', 'from diffusers.models.unets.unet_2d_blocks import (')
t = t.replace('from diffusers.models.dual_transformer_2d import DualTransformer2DModel', 'from diffusers.models.transformers.dual_transformer_2d import DualTransformer2DModel')
if 'from diffusers.models.normalization import AdaGroupNorm' not in t:
    t = t.replace('from diffusers.models.attention import AdaGroupNorm', 'try:\n    from diffusers.models.normalization import AdaGroupNorm\nexcept Exception:\n    from diffusers.models.attention import AdaGroupNorm')
write_if_changed(p, t)

p = ROOT / 'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py'
t = p.read_text(encoding='utf-8')
if 'from diffusers.utils.torch_utils import randn_tensor' not in t:
    t = re.sub(r'from diffusers\.utils import \((.*?)randn_tensor,?(.*?)\)', 'from diffusers.utils import (\\1\\2)\nfrom diffusers.utils.torch_utils import randn_tensor', t, flags=re.S)
    t = t.replace('from diffusers.utils import randn_tensor', 'from diffusers.utils.torch_utils import randn_tensor')
write_if_changed(p, t)

p = ROOT / 'automl/AutoencoderKL.py'
t = p.read_text(encoding='utf-8')
t = t.replace('from diffusers.models.vae import DecoderOutput, Decoder, DiagonalGaussianDistribution, Encoder', 'from diffusers.models.autoencoders.vae import DecoderOutput, Decoder, DiagonalGaussianDistribution, Encoder')
t = t.replace('from diffusers.models.vae import DecoderOutput, DiagonalGaussianDistribution, Encoder, Decoder', 'from diffusers.models.autoencoders.vae import DecoderOutput, DiagonalGaussianDistribution, Encoder, Decoder')
t = t.replace('from diffusers.utils import BaseOutput, randn_tensor', 'from diffusers.utils import BaseOutput\nfrom diffusers.utils.torch_utils import randn_tensor')
t = t.replace('from diffusers.models.modeling_utils import ModelMixin, apply_forward_hook', 'from diffusers.models.modeling_utils import ModelMixin\nfrom diffusers.utils.accelerate_utils import apply_forward_hook')
if 'from diffusers.loaders import FromOriginalVAEMixin' in t and 'single_file_model' not in t:
    t = t.replace('from diffusers.loaders import FromOriginalVAEMixin', 'try:\n    from diffusers.loaders import FromOriginalVAEMixin\nexcept Exception:\n    from diffusers.loaders.single_file_model import FromOriginalModelMixin as FromOriginalVAEMixin')
write_if_changed(p, t)
PY

python3 - <<'PY'
print('=== Fresh 5-module import test ===')
mods = {
    'UNetMV2DConditionModel': ('mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    'SingleImageDataset': ('mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    'MVDiffusionImagePipeline': ('mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    'AutoencoderKL': ('automl.AutoencoderKL', 'AutoencoderKL'),
    'save_cache': ('dataset.save_cache', 'save_cache'),
}
score = 0
for name, (mod, attr) in mods.items():
    try:
        m = __import__(mod, fromlist=[attr])
        getattr(m, attr)
        print(f'[PASS] {name}')
        score += 1
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'[FAIL] {name}: {type(e).__name__}: {e}')
print(f'Score: {score}/{len(mods)}')
PY

python3 - <<'PY'
from pathlib import Path
from huggingface_hub import snapshot_download
root = Path('/content/CADDreamer/ckpts/wonder3d-v1.0')
if not (root / 'model_index.json').exists():
    print('=== Download public Wonder3D checkpoint ===')
    out = snapshot_download(
        repo_id='flamehaze1115/wonder3d-v1.0',
        local_dir=str(root),
        local_dir_use_symlinks=False,
        resume_download=True,
    )
    print('downloaded to', out)
else:
    print('=== Wonder3D checkpoint already present ===')
    print(root)
PY%%bash
cat > /content/test_modules.py << 'HEREDOC'
import sys, os
sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')

print("=== CADDreamer Module Test ===")
mods = {
    'UNetMV2DConditionModel': 'mvdiffusion.models.unet_mv2d_condition',
        'SingleImageDataset': 'mvdiffusion.data.single_image_dataset',
            'MVDiffusionImagePipeline': 'mvdiffusion.pipelines.pipeline_mvdiffusion_image',
                'AutoencoderKL': 'automl.AutoencoderKL',
                }
                for name, mod in mods.items():
                    res = 'OK'
                        try:
                                __import__(mod)
                                    except Exception as e:
                                            res = str(e)[:80]
                                                print(f"  {name}: {res}")

                                                print("=== Repo Structure ===")
                                                for d in ['mvdiffusion', 'automl', 'neus', 'pyransac', 'utils']:
                                                    if os.path.exists(d):
                                                            files = [f for f in os.listdir(d) if f.endswith('.py')]
                                                                    print(f"  {d}/: {files[:5]}")
                                                                    print("Done!")
                                                                    HEREDOC
                                                                    cd /content/CADDreamer && python /content/test_modules.py 2>&1

In [ ]:
import os, re, sys, subprocess, traceback
from pathlib import Path

ROOT = Path('/content/CADDreamer')
if not (ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/lidan233/CADDreamer.git', str(ROOT)])

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print('=== Python / torch environment ===')
import torch
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

print('=== Install missing runtime packages ===')
for cmd in [
    [sys.executable, '-m', 'pip', 'install', '-q', 'trimesh', 'meshio', 'open3d', 'dill', 'rembg', 'onnxruntime', 'cupy-cuda12x'],
    [sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/facebookresearch/segment-anything.git'],
]:
    try:
        print('RUN', ' '.join(cmd[:6]), '...')
        subprocess.check_call(cmd)
    except Exception as e:
        print('install warning:', e)


def patch(path_str, transform):
    path = ROOT / path_str
    text = path.read_text(encoding='utf-8')
    new_text = transform(text)
    if new_text != text:
        path.write_text(new_text, encoding='utf-8')
        print('patched', path_str)
    else:
        print('ok', path_str)

patch('mvdiffusion/models/unet_mv2d_condition.py', lambda t: (
    t.replace('from diffusers.models.unet_2d_blocks import (', 'from diffusers.models.unets.unet_2d_blocks import (')
     .replace('from diffusers.models.dual_transformer_2d import DualTransformer2DModel', 'from diffusers.models.transformers.dual_transformer_2d import DualTransformer2DModel')
     .replace('from diffusers.models.modeling_utils import ModelMixin, load_state_dict, _load_state_dict_into_model', 'from diffusers.models.modeling_utils import ModelMixin\ntry:\n    from diffusers.models.modeling_utils import load_state_dict, _load_state_dict_into_model\nexcept Exception:\n    from diffusers.models.model_loading_utils import load_state_dict, _load_state_dict_into_model')
     .replace('from diffusers.utils import BaseOutput, logging, maybe_allow_in_graph', 'from diffusers.utils import BaseOutput, logging\ntry:\n    from diffusers.utils.torch_utils import maybe_allow_in_graph\nexcept Exception:\n    def maybe_allow_in_graph(cls):\n        return cls')
))

patch('mvdiffusion/models/unet_mv2d_blocks.py', lambda t: (
    t.replace('from diffusers.models.unet_2d_blocks import (', 'from diffusers.models.unets.unet_2d_blocks import (')
     .replace('from diffusers.models.dual_transformer_2d import DualTransformer2DModel', 'from diffusers.models.transformers.dual_transformer_2d import DualTransformer2DModel')
     .replace('from diffusers.models.attention import AdaGroupNorm', 'try:\n    from diffusers.models.normalization import AdaGroupNorm\nexcept Exception:\n    from diffusers.models.attention import AdaGroupNorm')
))

patch('mvdiffusion/pipelines/pipeline_mvdiffusion_image.py', lambda t: (
    t.replace('from diffusers.utils import deprecate, logging, randn_tensor', 'from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor')
     .replace('from diffusers.utils import randn_tensor', 'from diffusers.utils.torch_utils import randn_tensor')
))

patch('automl/AutoencoderKL.py', lambda t: (
    t.replace('from diffusers.models.vae import DecoderOutput, Decoder, DiagonalGaussianDistribution, Encoder', 'from diffusers.models.autoencoders.vae import DecoderOutput, Decoder, DiagonalGaussianDistribution, Encoder')
     .replace('from diffusers.models.vae import DecoderOutput, DiagonalGaussianDistribution, Encoder, Decoder', 'from diffusers.models.autoencoders.vae import DecoderOutput, DiagonalGaussianDistribution, Encoder, Decoder')
     .replace('from diffusers.utils import BaseOutput, apply_forward_hook', 'from diffusers.utils import BaseOutput\nfrom diffusers.utils.accelerate_utils import apply_forward_hook')
     .replace('from diffusers.utils import BaseOutput, randn_tensor', 'from diffusers.utils import BaseOutput\nfrom diffusers.utils.torch_utils import randn_tensor')
     .replace('from diffusers.utils import BaseOutput, apply_forward_hook, randn_tensor', 'from diffusers.utils import BaseOutput\nfrom diffusers.utils.accelerate_utils import apply_forward_hook\nfrom diffusers.utils.torch_utils import randn_tensor')
     .replace('from diffusers.loaders import FromOriginalVAEMixin', 'try:\n    from diffusers.loaders import FromOriginalVAEMixin\nexcept Exception:\n    from diffusers.loaders.single_file_model import FromOriginalModelMixin as FromOriginalVAEMixin')
))

print('=== Fresh 5-module import test ===')
mods = {
    'UNetMV2DConditionModel': ('mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    'SingleImageDataset': ('mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    'MVDiffusionImagePipeline': ('mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    'AutoencoderKL': ('automl.AutoencoderKL', 'AutoencoderKL'),
    'save_cache': ('save_cache', 'save_cache'),
}
score = 0
for name, (mod, attr) in mods.items():
    try:
        m = __import__(mod, fromlist=[attr])
        getattr(m, attr)
        print(f'[PASS] {name}')
        score += 1
    except Exception as e:
        traceback.print_exc()
        print(f'[FAIL] {name}: {type(e).__name__}: {e}')
print(f'Score: {score}/{len(mods)}')%%bash
# Test CADDreamer module imports using single-line python
cd /content/CADDreamer
python3 -c "import sys; sys.path.insert(0, '.'); mods=['mvdiffusion.models.unet_mv2d_condition', 'mvdiffusion.data.single_image_dataset', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'automl.AutoencoderKL']; [print(m.split('.')[-1]+': '+('OK' if not __import__('builtins').__dict__.get('_e') else _e)) or [print(m+': '+str(e)[:60]) for e in []] for m in mods]" 2>&1 | head -5 || true
python3 -c "
import sys
sys.path.insert(0, '.')
" 2>&1 || true
echo "--- Testing mvdiffusion imports ---"
python3 -c "import sys; sys.path.insert(0,'.');exec(open('mvdiffusion/__init__.py').read() if __import__('os').path.exists('mvdiffusion/__init__.py') else ''); print('mvdiffusion package found')" 2>&1 || true
echo "--- Repo files ---"
ls -la /content/CADDreamer/
echo "--- Cached outputs ---"
ls /content/CADDreamer/cached_output/cropsize-256-cfg1.0/0_0deepcad/ 2>&1 | head -20

In [ ]:
%%bash
cd /content && git clone https://github.com/lidan233/CADDreamer.git 2>&1 | tail -2
echo "=== Cached Output Files ===" && ls /content/CADDreamer/cached_output/cropsize-256-cfg1.0/0_0deepcad/

In [ ]:
import os, sys, subprocess, traceback
from pathlib import Path

ROOT = Path('/content/CADDreamer')
if not (ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/lidan233/CADDreamer.git', str(ROOT)])

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print('=== Python / torch environment ===')
import torch
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

print('=== Install missing runtime packages ===')
commands = [
    [sys.executable, '-m', 'pip', 'install', '-q', 'trimesh', 'meshio', 'open3d', 'dill', 'rembg', 'onnxruntime', 'cupy-cuda12x'],
    [sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/facebookresearch/segment-anything.git'],
]
for cmd in commands:
    try:
        subprocess.check_call(cmd)
        print('ok:', ' '.join(cmd[3:]))
    except Exception as e:
        print('install warning:', e)


def patch(rel_path, replacements):
    path = ROOT / rel_path
    text = path.read_text(encoding='utf-8')
    orig = text
    for old, new in replacements:
        text = text.replace(old, new)
    if text != orig:
        path.write_text(text, encoding='utf-8')
        print('patched', rel_path)
    else:
        print('ok', rel_path)

patch('mvdiffusion/models/unet_mv2d_condition.py', [
    ('from diffusers.models.unet_2d_blocks import (', 'from diffusers.models.unets.unet_2d_blocks import ('),
    ('from diffusers.models.dual_transformer_2d import DualTransformer2DModel', 'from diffusers.models.transformers.dual_transformer_2d import DualTransformer2DModel'),
    ('from diffusers.models.modeling_utils import ModelMixin, load_state_dict, _load_state_dict_into_model', 'from diffusers.models.modeling_utils import ModelMixin\ntry:\n    from diffusers.models.modeling_utils import load_state_dict, _load_state_dict_into_model\nexcept Exception:\n    from diffusers.models.model_loading_utils import load_state_dict, _load_state_dict_into_model'),
    ('from diffusers.utils import BaseOutput, logging, maybe_allow_in_graph', 'from diffusers.utils import BaseOutput, logging\ntry:\n    from diffusers.utils.torch_utils import maybe_allow_in_graph\nexcept Exception:\n    def maybe_allow_in_graph(cls):\n        return cls'),
])

patch('mvdiffusion/models/unet_mv2d_blocks.py', [
    ('from diffusers.models.unet_2d_blocks import (', 'from diffusers.models.unets.unet_2d_blocks import ('),
    ('from diffusers.models.dual_transformer_2d import DualTransformer2DModel', 'from diffusers.models.transformers.dual_transformer_2d import DualTransformer2DModel'),
    ('from diffusers.models.attention import AdaGroupNorm', 'try:\n    from diffusers.models.normalization import AdaGroupNorm\nexcept Exception:\n    from diffusers.models.attention import AdaGroupNorm'),
])

patch('mvdiffusion/pipelines/pipeline_mvdiffusion_image.py', [
    ('from diffusers.utils import deprecate, logging, randn_tensor', 'from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor'),
    ('from diffusers.utils import randn_tensor', 'from diffusers.utils.torch_utils import randn_tensor'),
])

patch('automl/AutoencoderKL.py', [
    ('from diffusers.models.vae import DecoderOutput, Decoder, DiagonalGaussianDistribution, Encoder', 'from diffusers.models.autoencoders.vae import DecoderOutput, Decoder, DiagonalGaussianDistribution, Encoder'),
    ('from diffusers.models.vae import DecoderOutput, DiagonalGaussianDistribution, Encoder, Decoder', 'from diffusers.models.autoencoders.vae import DecoderOutput, DiagonalGaussianDistribution, Encoder, Decoder'),
    ('from diffusers.utils import BaseOutput, apply_forward_hook, randn_tensor', 'from diffusers.utils import BaseOutput\nfrom diffusers.utils.accelerate_utils import apply_forward_hook\nfrom diffusers.utils.torch_utils import randn_tensor'),
    ('from diffusers.utils import BaseOutput, apply_forward_hook', 'from diffusers.utils import BaseOutput\nfrom diffusers.utils.accelerate_utils import apply_forward_hook'),
    ('from diffusers.utils import BaseOutput, randn_tensor', 'from diffusers.utils import BaseOutput\nfrom diffusers.utils.torch_utils import randn_tensor'),
    ('from diffusers.loaders import FromOriginalVAEMixin', 'try:\n    from diffusers.loaders import FromOriginalVAEMixin\nexcept Exception:\n    from diffusers.loaders.single_file_model import FromOriginalModelMixin as FromOriginalVAEMixin'),
])

print('=== Fresh 5-module import test ===')
mods = {
    'UNetMV2DConditionModel': ('mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    'SingleImageDataset': ('mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    'MVDiffusionImagePipeline': ('mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    'AutoencoderKL': ('automl.AutoencoderKL', 'AutoencoderKL'),
    'save_cache': ('save_cache', 'save_cache'),
}
score = 0
for name, (mod, attr) in mods.items():
    try:
        m = __import__(mod, fromlist=[attr])
        getattr(m, attr)
        print(f'[PASS] {name}')
        score += 1
    except Exception as e:
        traceback.print_exc()
        print(f'[FAIL] {name}: {type(e).__name__}: {e}')
print(f'Score: {score}/{len(mods)}')

In [ ]:
%%bash
# Step 2: Install Python Dependencies
# CADDreamer requires specific versions - install compatible versions for Colab's PyTorch
pip install -q diffusers==0.19.3 transformers==4.44.0 accelerate==0.33.0 huggingface-hub==0.24.5
pip install -q einops omegaconf pytorch-lightning==1.9.5
pip install -q torchmetrics==1.4.1
pip install -q scikit-image==0.25.0 imageio==2.34.2
pip install -q tqdm requests pillow
pip install -q xformers --index-url https://download.pytorch.org/whl/cu118
echo "Dependencies installed!"
python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available())"

In [ ]:
%%bash
# Step 3: Install additional dependencies needed for CADDreamer
pip install -q rembg onnxruntime
pip install -q packaging omegaconf einops
pip install -q open3d trimesh
pip install -q matplotlib pillow numpy
pip install -q pytorch-lightning==1.9.5 --quiet 2>/dev/null || true
echo "Additional packages installed!"
python -c "import diffusers; print('diffusers:', diffusers.__version__)"
python -c "import transformers; print('transformers:', transformers.__version__)"

In [ ]:
from pathlib import Path
import traceback
ROOT = Path('/content/CADDreamer')
print('ROOT exists:', ROOT.exists())
for rel in [
    'automl/AutoencoderKL.py',
    'mvdiffusion/models/unet_mv2d_condition.py',
    'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py',
    'mvdiffusion/data/single_image_dataset.py',
]:
    p = ROOT / rel
    print('\n===', rel, 'exists=', p.exists(), '===')
    if p.exists():
        lines = p.read_text(encoding='utf-8', errors='ignore').splitlines()
        for i, line in enumerate(lines[:40], 1):
            print(f'{i:02d}: {line}')

print('\n=== save_cache candidates ===')
for p in sorted(ROOT.glob('**/*save_cache*.py'))[:50]:
    print(p)

print('\n=== package versions/imports ===')
for mod in ['numpy', 'scipy', 'PIL', 'torchvision', 'diffusers', 'transformers', 'numba', 'segment_anything', 'trimesh']:
    try:
        m = __import__(mod)
        print(mod, getattr(m, '__version__', 'no __version__'))
    except Exception as e:
        print(mod, 'IMPORT_FAIL', type(e).__name__, e)%%bash
# Step 4: Fix JAX/diffusers compatibility + upgrade diffusers to a working version
# The issue is diffusers 0.19.3 tries to import flax which conflicts with newer JAX
# We'll use a newer diffusers that doesn't have this issue
pip install -q "diffusers>=0.27.0" 2>/dev/null
pip install -q jax==0.4.26 jaxlib==0.4.26+cuda12.cudnn89 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html 2>/dev/null || true
echo "=== Checking key imports ==="
python -c "import torch; print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"
python -c "import diffusers; print('diffusers:', diffusers.__version__)" 2>&1 | head -3
python -c "import transformers; print('transformers:', transformers.__version__)" 2>&1 | head -3
echo "Done!"

In [ ]:
%%bash
# Step 5: Fix transformers compatibility
pip install -q "transformers>=4.35.0" 2>/dev/null
pip install -q tokenizers 2>/dev/null
echo "=== Testing imports ==="
python -c "
import sys
sys.path.insert(0, '/content/CADDreamer')
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
import diffusers
print('diffusers:', diffusers.__version__)
import transformers
print('transformers:', transformers.__version__)
import einops
print('einops:', einops.__version__)
import omegaconf
print('omegaconf: OK')
print('All key imports successful!')
" 2>&1

In [ ]:
%%bash
# Step 6: Set up CADDreamer Python path and test custom module imports
cd /content/CADDreamer
python -c "
import sys
sys.path.insert(0, '/content/CADDreamer')
print('=== Testing CADDreamer module imports ===')

# Test mvdiffusion modules
try:
    from mvdiffusion.models.unet_mv2d_condition import UNetMV2DConditionModel
        print('UNetMV2DConditionModel: OK')
        except Exception as e:
            print('UNetMV2DConditionModel ERROR:', str(e)[:100])

            try:
                from mvdiffusion.data.single_image_dataset import SingleImageDataset
                    print('SingleImageDataset: OK')
                    except Exception as e:
                        print('SingleImageDataset ERROR:', str(e)[:100])

                        try:
                            from mvdiffusion.pipelines.pipeline_mvdiffusion_image import MVDiffusionImagePipeline
                                print('MVDiffusionImagePipeline: OK')
                                except Exception as e:
                                    print('MVDiffusionImagePipeline ERROR:', str(e)[:100])

                                    try:
                                        from automl.AutoencoderKL import AutoencoderKL, ClassifyMLP
                                            print('AutoencoderKL, ClassifyMLP: OK')
                                            except Exception as e:
                                                print('AutoencoderKL ERROR:', str(e)[:100])

                                                try:
                                                    from utils.util import save_cache
                                                        print('utils.util: OK')
                                                        except Exception as e:
                                                            print('utils.util ERROR:', str(e)[:100])
                                                            " 2>&1

In [ ]:
import sys
import os

# Step 6: Set up Python path for CADDreamer
sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')

print("=== Testing CADDreamer module imports ===")
print(f"Current dir: {os.getcwd()}")

In [ ]:
%%writefile /content/test_imports.py
import sys
sys.path.insert(0, '/content/CADDreamer')
print("=== Testing CADDreamer module imports ===")

try:
    from mvdiffusion.models.unet_mv2d_condition import UNetMV2DConditionModel
        print("UNetMV2DConditionModel: OK")
        except Exception as e:
            print(f"UNetMV2DConditionModel ERROR: {str(e)[:150]}")

            try:
                from mvdiffusion.data.single_image_dataset import SingleImageDataset
                    print("SingleImageDataset: OK")
                    except Exception as e:
                        print(f"SingleImageDataset ERROR: {str(e)[:150]}")

                        try:
                            from mvdiffusion.pipelines.pipeline_mvdiffusion_image import MVDiffusionImagePipeline
                                print("MVDiffusionImagePipeline: OK")
                                except Exception as e:
                                    print(f"MVDiffusionImagePipeline ERROR: {str(e)[:150]}")

                                    try:
                                        from automl.AutoencoderKL import AutoencoderKL, ClassifyMLP
                                            print("AutoencoderKL, ClassifyMLP: OK")
                                            except Exception as e:
                                                print(f"AutoencoderKL ERROR: {str(e)[:150]}")

                                                try:
                                                    from utils.util import save_cache
                                                        print("utils.util: OK")
                                                        except Exception as e:
                                                            print(f"utils.util ERROR: {str(e)[:150]}")

                                                            print("Import test complete!")

In [ ]:
%%bash
cd /content/CADDreamer
python /content/test_imports.py 2>&1

In [ ]:
script = """import sys
sys.path.insert(0, '/content/CADDreamer')
results = {}
try:
    from mvdiffusion.models.unet_mv2d_condition import UNetMV2DConditionModel
        results['UNetMV2DConditionModel'] = 'OK'
        except Exception as e:
            results['UNetMV2DConditionModel'] = 'ERROR: ' + str(e)[:100]
            try:
                from mvdiffusion.data.single_image_dataset import SingleImageDataset
                    results['SingleImageDataset'] = 'OK'
                    except Exception as e:
                        results['SingleImageDataset'] = 'ERROR: ' + str(e)[:100]
                        try:
                            from mvdiffusion.pipelines.pipeline_mvdiffusion_image import MVDiffusionImagePipeline
                                results['MVDiffusionImagePipeline'] = 'OK'
                                except Exception as e:
                                    results['MVDiffusionImagePipeline'] = 'ERROR: ' + str(e)[:100]
                                    try:
                                        from automl.AutoencoderKL import AutoencoderKL, ClassifyMLP
                                            results['AutoencoderKL'] = 'OK'
                                            except Exception as e:
                                                results['AutoencoderKL'] = 'ERROR: ' + str(e)[:100]
                                                try:
                                                    from utils.util import save_cache
                                                        results['utils'] = 'OK'
                                                        except Exception as e:
                                                            results['utils'] = 'ERROR: ' + str(e)[:100]
                                                            for k, v in results.items():
                                                                print(k + ': ' + v)
                                                                print('Done!')
                                                                """
                                                                with open('/content/test_imports.py', 'w') as f:
                                                                    f.write(script)
                                                                    print("Script written successfully!")

In [ ]:
%%bash
# Step 7: Test CADDreamer imports
if [ ! -d /content/CADDreamer ]; then
  cd /content && git clone https://github.com/lidan233/CADDreamer.git --quiet
fi
pip install -q "diffusers>=0.27.0" "transformers>=4.35.0" 2>/dev/null

# Write Python test script line by line (no heredoc)
python3 << EOF
import sys
sys.path.insert(0, "/content/CADDreamer")
import os
os.chdir("/content/CADDreamer")

results = {}

try:
    import torch
    results["torch"] = "OK - " + torch.__version__ + " CUDA:" + str(torch.cuda.is_available())
except Exception as e:
    results["torch"] = "ERR: " + str(e)[:60]

try:
    import diffusers
    results["diffusers"] = "OK - " + diffusers.__version__
except Exception as e:
    results["diffusers"] = "ERR: " + str(e)[:60]

try:
    import transformers
    results["transformers"] = "OK - " + transformers.__version__
except Exception as e:
    results["transformers"] = "ERR: " + str(e)[:60]

try:
    from mvdiffusion.models.unet_mv2d_condition import UNetMV2DConditionModel
    results["UNetMV2DConditionModel"] = "OK"
except Exception as e:
    results["UNetMV2DConditionModel"] = "ERR: " + str(e)[:60]

try:
    from mvdiffusion.data.single_image_dataset import SingleImageDataset
    results["SingleImageDataset"] = "OK"
except Exception as e:
    results["SingleImageDataset"] = "ERR: " + str(e)[:60]

try:
    from mvdiffusion.pipelines.pipeline_mvdiffusion_image import MVDiffusionImagePipeline
    results["MVDiffusionImagePipeline"] = "OK"
except Exception as e:
    results["MVDiffusionImagePipeline"] = "ERR: " + str(e)[:60]

try:
    from automl.AutoencoderKL import AutoencoderKL
    results["AutoencoderKL"] = "OK"
except Exception as e:
    results["AutoencoderKL"] = "ERR: " + str(e)[:60]

print("=== CADDreamer Module Import Results ===")
for k, v in results.items():
    status = "PASS" if v.startswith("OK") else "FAIL"
    print("  [" + status + "] " + k + ": " + v)

cache_dir = "/content/CADDreamer/cached_output/cropsize-256-cfg1.0/0_0deepcad"
if os.path.exists(cache_dir):
    files = sorted(os.listdir(cache_dir))
    print("\n=== Cached Output Files (" + str(len(files)) + " files) ===")
    for f in files:
        sz = os.path.getsize(os.path.join(cache_dir, f))
        print("  " + f + " (" + str(sz//1024) + "KB)")
else:
    print("Cache dir not found")
EOF

In [ ]:
%%bash
# Step 8: Fix dependency compatibility for CADDreamer
# Key issues:
# 1. randn_tensor removed from diffusers.utils in 0.27+
# 2. _load_state_dict_into_model removed in newer diffusers
# 3. segment_anything not installed
# 4. FromOriginalVAEMixin not available

# Install segment-anything
pip install -q segment-anything 2>/dev/null

# Downgrade diffusers to 0.21.4 (compatible with CADDreamer)
pip install -q "diffusers==0.21.4" 2>/dev/null

# Verify
python3 -c "import diffusers; print('diffusers:', diffusers.__version__)"
python3 -c "import segment_anything; print('segment_anything: OK')" 2>&1 || echo "segment_anything: FAILED"

In [ ]:
%%bash
# Step 9: Fix huggingface_hub + diffusers compatibility
# diffusers 0.21.4 needs older huggingface_hub (<0.20)
# But we can use a different approach: install diffusers 0.24.x which is more compatible

# First check what huggingface_hub version we have
python3 -c "import huggingface_hub; print('hf_hub:', huggingface_hub.__version__)"

# Try diffusers 0.24.0 (released Dec 2023, should be compatible)
pip install -q "diffusers==0.24.0" "huggingface_hub==0.20.3" 2>/dev/null

# Verify
python3 -c "import diffusers; print('diffusers:', diffusers.__version__)"
python3 -c "import huggingface_hub; print('hf_hub:', huggingface_hub.__version__)"

In [ ]:
%%bash
# Step 10: Re-test CADDreamer module imports with fixed dependencies
cd /content/CADDreamer
python3 << EOF
import sys
sys.path.insert(0, "/content/CADDreamer")
import os
os.chdir("/content/CADDreamer")

results = {}

try:
    import torch
    results["torch"] = "OK - " + torch.__version__ + " CUDA:" + str(torch.cuda.is_available())
except Exception as e:
    results["torch"] = "ERR: " + str(e)[:80]

try:
    import diffusers
    results["diffusers"] = "OK - " + diffusers.__version__
except Exception as e:
    results["diffusers"] = "ERR: " + str(e)[:80]

try:
    import transformers
    results["transformers"] = "OK - " + transformers.__version__
except Exception as e:
    results["transformers"] = "ERR: " + str(e)[:80]

try:
    from mvdiffusion.models.unet_mv2d_condition import UNetMV2DConditionModel
    results["UNetMV2DConditionModel"] = "OK"
except Exception as e:
    results["UNetMV2DConditionModel"] = "ERR: " + str(e)[:80]

try:
    from mvdiffusion.data.single_image_dataset import SingleImageDataset
    results["SingleImageDataset"] = "OK"
except Exception as e:
    results["SingleImageDataset"] = "ERR: " + str(e)[:80]

try:
    from mvdiffusion.pipelines.pipeline_mvdiffusion_image import MVDiffusionImagePipeline
    results["MVDiffusionImagePipeline"] = "OK"
except Exception as e:
    results["MVDiffusionImagePipeline"] = "ERR: " + str(e)[:80]

try:
    from automl.AutoencoderKL import AutoencoderKL
    results["AutoencoderKL"] = "OK"
except Exception as e:
    results["AutoencoderKL"] = "ERR: " + str(e)[:80]

try:
    from utils.util import save_cache
    results["utils"] = "OK"
except Exception as e:
    results["utils"] = "ERR: " + str(e)[:80]

print("=== CADDreamer Module Import Results ===")
for k, v in results.items():
    status = "PASS" if v.startswith("OK") else "FAIL"
    print("  [" + status + "] " + k + ": " + v)
EOF

In [ ]:
%%bash
# Step 11: Final fix - install latest diffusers + missing packages + patch CADDreamer

# Restore latest diffusers (0.38.0 had most features)
pip install -q "diffusers==0.29.2" "huggingface_hub>=0.23.0" "transformers>=4.35.0" 2>/dev/null

# Install missing packages
pip install -q rembg trimesh open3d 2>/dev/null

# Check what functions are available in this diffusers version
python3 -c "import diffusers; print('diffusers:', diffusers.__version__); from diffusers.utils import randn_tensor; print('randn_tensor: OK')" 2>&1
python3 -c "from diffusers.loaders import AttnProcsLayers; print('AttnProcsLayers: OK')" 2>&1

# Now patch CADDreamer source for compatibility
# Fix 1: unet_mv2d_condition.py - _load_state_dict_into_model removed in 0.25+
sed -i "s/from diffusers.loaders.unet import/# from diffusers.loaders.unet import/" /content/CADDreamer/mvdiffusion/models/unet_mv2d_condition.py 2>/dev/null || echo "sed 1 skipped"

# Check what imports are in unet_mv2d_condition.py
head -20 /content/CADDreamer/mvdiffusion/models/unet_mv2d_condition.py

In [ ]:
%%bash
# Step 12: Patch CADDreamer source files for diffusers 0.29.2 compatibility

# 1. Fix randn_tensor import in pipeline file
grep -n "randn_tensor" /content/CADDreamer/mvdiffusion/pipelines/pipeline_mvdiffusion_image.py | head -5
sed -i "s/from diffusers.utils import.*randn_tensor.*/from diffusers.utils.torch_utils import randn_tensor/" /content/CADDreamer/mvdiffusion/pipelines/pipeline_mvdiffusion_image.py
grep -n "randn_tensor" /content/CADDreamer/mvdiffusion/pipelines/pipeline_mvdiffusion_image.py | head -5

# 2. Check what _load_state_dict_into_model is used for in unet
grep -n "_load_state_dict\|loaders.unet" /content/CADDreamer/mvdiffusion/models/unet_mv2d_condition.py | head -10

# 3. Fix the UNet imports - _load_state_dict_into_model is now in diffusers.loaders.single_file_utils
python3 -c "from diffusers.loaders import UNet2DConditionLoadersMixin; print('UNet2DConditionLoadersMixin: OK')" 2>&1

# 4. Also check automl/AutoencoderKL.py for apply_forward_hook issue
grep -n "apply_forward_hook\|FromOriginalVAE" /content/CADDreamer/automl/AutoencoderKL.py | head -10

# 5. Verify rembg and trimesh are installed
python3 -c "import rembg; print('rembg: OK')" 2>&1
python3 -c "import trimesh; print('trimesh:', trimesh.__version__)" 2>&1

In [ ]:
%%bash
# Step 13: Final patches for remaining issues

# 1. Check if apply_forward_hook exists in diffusers 0.29.2
python3 -c "from diffusers.utils import apply_forward_hook; print('apply_forward_hook: OK')" 2>&1
python3 -c "from diffusers.utils.accelerate_utils import apply_forward_hook; print('accel apply_forward_hook: OK')" 2>&1

# 2. Install rembg with onnxruntime (for background removal)
pip install -q onnxruntime 2>/dev/null
python3 -c "import rembg; print('rembg: OK')" 2>&1

# 3. Check if _load_state_dict_into_model exists in diffusers.models.modeling_utils
python3 -c "from diffusers.models.modeling_utils import _load_state_dict_into_model; print('_load_state_dict_into_model: OK')" 2>&1

# 4. Check FromOriginalVAEMixin
python3 -c "from diffusers.loaders import FromOriginalVAEMixin; print('FromOriginalVAEMixin: OK')" 2>&1

In [ ]:
%%bash
# Step 14: Apply final patches to AutoencoderKL.py

# Fix 1: apply_forward_hook - move from diffusers.utils to diffusers.utils.accelerate_utils
sed -i "s/from diffusers.utils import BaseOutput, apply_forward_hook/from diffusers.utils import BaseOutput\nfrom diffusers.utils.accelerate_utils import apply_forward_hook/" /content/CADDreamer/automl/AutoencoderKL.py

# Fix 2: FromOriginalVAEMixin - not in 0.29.2, create a stub
python3 -c "import diffusers.loaders as L; print(dir(L))" 2>&1 | grep -i "vae\|original\|mixin" | head -5

# Fix 3: Remove FromOriginalVAEMixin from class definition (it might not be needed for inference)
sed -i "s/from diffusers.loaders import FromOriginalVAEMixin/# from diffusers.loaders import FromOriginalVAEMixin\nFromOriginalVAEMixin = object/" /content/CADDreamer/automl/AutoencoderKL.py

# Verify patches
head -25 /content/CADDreamer/automl/AutoencoderKL.py

In [ ]:
%%bash
# Step 15: Final comprehensive import test for all CADDreamer modules
cd /content/CADDreamer
python3 << EOF
import sys
sys.path.insert(0, "/content/CADDreamer")
import os
os.chdir("/content/CADDreamer")

results = {}

try:
    import torch
    results["torch"] = "OK - " + torch.__version__ + " CUDA:" + str(torch.cuda.is_available())
except Exception as e:
    results["torch"] = "ERR: " + str(e)[:80]

try:
    import diffusers
    results["diffusers"] = "OK - " + diffusers.__version__
except Exception as e:
    results["diffusers"] = "ERR: " + str(e)[:80]

try:
    import transformers
    results["transformers"] = "OK - " + transformers.__version__
except Exception as e:
    results["transformers"] = "ERR: " + str(e)[:80]

try:
    from mvdiffusion.models.unet_mv2d_condition import UNetMV2DConditionModel
    results["UNetMV2DConditionModel"] = "OK"
except Exception as e:
    results["UNetMV2DConditionModel"] = "ERR: " + str(e)[:80]

try:
    from mvdiffusion.data.single_image_dataset import SingleImageDataset
    results["SingleImageDataset"] = "OK"
except Exception as e:
    results["SingleImageDataset"] = "ERR: " + str(e)[:80]

try:
    from mvdiffusion.pipelines.pipeline_mvdiffusion_image import MVDiffusionImagePipeline
    results["MVDiffusionImagePipeline"] = "OK"
except Exception as e:
    results["MVDiffusionImagePipeline"] = "ERR: " + str(e)[:80]

try:
    from automl.AutoencoderKL import AutoencoderKL
    results["AutoencoderKL"] = "OK"
except Exception as e:
    results["AutoencoderKL"] = "ERR: " + str(e)[:80]

try:
    from utils.util import save_cache
    results["utils"] = "OK"
except Exception as e:
    results["utils"] = "ERR: " + str(e)[:80]

print("=== FINAL CADDreamer Module Import Results ===")
passed = sum(1 for v in results.values() if v.startswith("OK"))
total = len(results)
print("  Score: " + str(passed) + "/" + str(total) + " modules OK")
for k, v in results.items():
    status = "PASS" if v.startswith("OK") else "FAIL"
    print("  [" + status + "] " + k + ": " + v)
EOF

In [ ]:
%%bash
# Step 16: Fix remaining compatibility issues

# 1. Install meshio
pip install -q meshio 2>/dev/null

# 2. Check what modules are where in diffusers 0.29.2
python3 -c "from diffusers.models import UNet2DConditionModel; print('UNet2DConditionModel OK')" 2>&1
python3 -c "from diffusers.models.unets import unet_2d_condition; print('unets.unet_2d_condition OK')" 2>&1

# 3. Fix UNet import - unet_2d_blocks was moved
grep -n "unet_2d_blocks" /content/CADDreamer/mvdiffusion/models/unet_mv2d_condition.py | head -5
sed -i "s/from diffusers.models.unet_2d_blocks import/from diffusers.models.unets.unet_2d_blocks import/" /content/CADDreamer/mvdiffusion/models/unet_mv2d_condition.py 2>/dev/null || true
sed -i "s/from diffusers.models import unet_2d_blocks/from diffusers.models.unets import unet_2d_blocks/" /content/CADDreamer/mvdiffusion/models/unet_mv2d_condition.py 2>/dev/null || true

# 4. Check unet_2d_blocks path
python3 -c "import diffusers.models.unets.unet_2d_blocks; print('unet_2d_blocks found at unets')" 2>&1
python3 -c "import diffusers.models.unet_2d_blocks; print('unet_2d_blocks found at models')" 2>&1

# 5. Fix FLAX_WEIGHTS_NAME in pipeline
grep -n "FLAX_WEIGHTS" /content/CADDreamer/mvdiffusion/pipelines/pipeline_mvdiffusion_image.py | head -5
sed -i "/FLAX_WEIGHTS_NAME/d" /content/CADDreamer/mvdiffusion/pipelines/pipeline_mvdiffusion_image.py 2>/dev/null || true

# 6. Fix diffusers.models.vae for AutoencoderKL
grep -n "diffusers.models.vae" /content/CADDreamer/automl/AutoencoderKL.py | head -5
python3 -c "from diffusers.models.autoencoders.autoencoder_kl import DiagonalGaussianDistribution; print('DiagonalGaussianDistribution OK')" 2>&1

In [ ]:
%%bash
# Step 17: Final critical patches

# 1. Fix unet_mv2d_condition.py - unet_2d_blocks moved to unets subpackage
sed -i "s|from diffusers.models.unet_2d_blocks import|from diffusers.models.unets.unet_2d_blocks import|g" /content/CADDreamer/mvdiffusion/models/unet_mv2d_condition.py
echo "UNet patch applied"
grep -n "unet_2d_blocks" /content/CADDreamer/mvdiffusion/models/unet_mv2d_condition.py | head -3

# 2. Fix AutoencoderKL.py - diffusers.models.vae moved to diffusers.models.autoencoders
python3 -c "from diffusers.models.autoencoders.autoencoder_kl import AutoencoderKL as DiffAE; print('AutoencoderKL found at autoencoders')" 2>&1
python3 -c "from diffusers.models import AutoencoderKL as DiffAE; print('AutoencoderKL found at models')" 2>&1

# Check what is in diffusers.models.vae
python3 -c "import diffusers.models.vae; print('vae module OK')" 2>&1

# If diffusers.models.vae doesnt exist, create a compatibility shim
python3 -c "from diffusers.models import Decoder, Encoder; print('Decoder,Encoder from models OK')" 2>&1
python3 -c "from diffusers.models.autoencoders import AutoencoderKL; d=dir(AutoencoderKL); print([x for x in d if not x.startswith('_')][:10])" 2>&1

# 3. Fix pipeline FLAX_WEIGHTS_NAME
grep -n "FLAX_WEIGHTS" /content/CADDreamer/mvdiffusion/pipelines/pipeline_mvdiffusion_image.py | head -3

In [ ]:
%%bash
# Step 18: Fix AutoencoderKL.py vae import + utils meshio + pipeline FLAX_WEIGHTS

# Check what AutoencoderKL.py imports from vae
grep -n "models.vae" /content/CADDreamer/automl/AutoencoderKL.py | head -10

# Fix: replace diffusers.models.vae with diffusers.models.autoencoders.autoencoder_kl
sed -i "s|from diffusers.models.vae import|from diffusers.models.autoencoders.autoencoder_kl import|g" /content/CADDreamer/automl/AutoencoderKL.py

# Verify the fix
grep -n "from diffusers.models" /content/CADDreamer/automl/AutoencoderKL.py | head -5

# Check if the vae imports work now
python3 -c "from diffusers.models.autoencoders.autoencoder_kl import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder; print('Decoder,DecoderOutput,DGD,Encoder: OK')" 2>&1

# Fix utils - install meshio
pip install -q meshio 2>/dev/null
python3 -c "import meshio; print('meshio:', meshio.__version__)" 2>&1

# Check FLAX_WEIGHTS_NAME - should have been deleted in previous step
grep -n "FLAX_WEIGHTS" /content/CADDreamer/mvdiffusion/pipelines/pipeline_mvdiffusion_image.py | head -3
echo "If no output above, FLAX_WEIGHTS_NAME already removed"

In [ ]:
%%bash
# Step 19: Fix remaining unet_2d_blocks imports in AutoencoderKL.py + comprehensive re-test

# Fix AutoencoderKL.py unet_2d_blocks import
sed -i "s|from diffusers.models.unet_2d_blocks import|from diffusers.models.unets.unet_2d_blocks import|g" /content/CADDreamer/automl/AutoencoderKL.py
echo "AutoencoderKL unet_2d_blocks patch applied"

# Also fix any remaining references in mvdiffusion
grep -rl "from diffusers.models.unet_2d_blocks" /content/CADDreamer/ 2>/dev/null | head -5
for f in $(grep -rl "from diffusers.models.unet_2d_blocks" /content/CADDreamer/ 2>/dev/null); do
    sed -i "s|from diffusers.models.unet_2d_blocks import|from diffusers.models.unets.unet_2d_blocks import|g" "$f"
    echo "Fixed: $f"
done

# Check FLAX_WEIGHTS_NAME in pipeline file
grep -n "FLAX_WEIGHTS" /content/CADDreamer/mvdiffusion/pipelines/pipeline_mvdiffusion_image.py | head -3
echo "Pipeline FLAX check done"

# Now run the comprehensive import test
cd /content/CADDreamer
python3 << EOF
import sys
sys.path.insert(0, "/content/CADDreamer")
import os
os.chdir("/content/CADDreamer")

results = {}

try:
    import torch
    results["torch"] = "OK - " + torch.__version__
except Exception as e:
    results["torch"] = "ERR: " + str(e)[:60]

try:
    from mvdiffusion.models.unet_mv2d_condition import UNetMV2DConditionModel
    results["UNetMV2DConditionModel"] = "OK"
except Exception as e:
    results["UNetMV2DConditionModel"] = "ERR: " + str(e)[:80]

try:
    from mvdiffusion.data.single_image_dataset import SingleImageDataset
    results["SingleImageDataset"] = "OK"
except Exception as e:
    results["SingleImageDataset"] = "ERR: " + str(e)[:80]

try:
    from mvdiffusion.pipelines.pipeline_mvdiffusion_image import MVDiffusionImagePipeline
    results["MVDiffusionImagePipeline"] = "OK"
except Exception as e:
    results["MVDiffusionImagePipeline"] = "ERR: " + str(e)[:80]

try:
    from automl.AutoencoderKL import AutoencoderKL
    results["AutoencoderKL"] = "OK"
except Exception as e:
    results["AutoencoderKL"] = "ERR: " + str(e)[:80]

try:
    from utils.util import save_cache
    results["utils"] = "OK"
except Exception as e:
    results["utils"] = "ERR: " + str(e)[:80]

print("=== UPDATED IMPORT TEST ===")
passed = sum(1 for v in results.values() if v.startswith("OK"))
print("Score: " + str(passed) + "/" + str(len(results)))
for k, v in results.items():
    s = "PASS" if v.startswith("OK") else "FAIL"
    print("  [" + s + "] " + k + ": " + v)
EOF

In [ ]:

%%bash
# Step 20: Fix remaining import errors - DIFFUSERS_CACHE, FLAX_WEIGHTS_NAME, randn_tensor
cd /content/CADDreamer

echo "=== Fix 1: DIFFUSERS_CACHE ==="
grep -n "DIFFUSERS_CACHE" mvdiffusion/models/unet_mv2d_condition.py | head -3
python3 - << 'PYEOF'
content = open('mvdiffusion/models/unet_mv2d_condition.py').read()
if 'DIFFUSERS_CACHE' in content and 'except ImportError' not in content[:500]:
    shim = '\ntry:\n    from diffusers.utils import DIFFUSERS_CACHE\nexcept ImportError:\n    import os\n    DIFFUSERS_CACHE = os.path.expanduser("~/.cache/huggingface/diffusers")\n'
    idx = content.find('from diffusers')
    nl = content.find('\n', idx)
    content = content[:nl+1] + shim + content[nl+1:]
    open('mvdiffusion/models/unet_mv2d_condition.py', 'w').write(content)
    print('DIFFUSERS_CACHE shim added')
else:
    print('DIFFUSERS_CACHE already fixed or not found')
PYEOF

echo "=== Fix 2: FLAX_WEIGHTS_NAME ==="
grep -n "FLAX_WEIGHTS_NAME" mvdiffusion/pipelines/pipeline_mvdiffusion_image.py | head -3
python3 - << 'PYEOF'
import re
content = open('mvdiffusion/pipelines/pipeline_mvdiffusion_image.py').read()
if 'FLAX_WEIGHTS_NAME' in content:
    content = re.sub(r'from transformers\.utils import[^\n]*FLAX_WEIGHTS_NAME[^\n]*\n', 'FLAX_WEIGHTS_NAME = "flax_model.msgpack"\n', content)
    open('mvdiffusion/pipelines/pipeline_mvdiffusion_image.py', 'w').write(content)
    print('FLAX_WEIGHTS_NAME fixed')
else:
    print('FLAX_WEIGHTS_NAME not found')
PYEOF

echo "=== Fix 3: randn_tensor in AutoencoderKL ==="
grep -n "randn_tensor" automl/AutoencoderKL.py | head -3
python3 - << 'PYEOF'
content = open('automl/AutoencoderKL.py').read()
if 'randn_tensor' in content and 'torch_utils' not in content:
    shim = '\ntry:\n    from diffusers.utils.torch_utils import randn_tensor\nexcept ImportError:\n    import torch\n    def randn_tensor(shape, generator=None, device=None, dtype=None):\n        return torch.randn(shape, generator=generator, device=device, dtype=dtype)\n'
    idx = content.find('from diffusers')
    nl = content.find('\n', idx)
    content = content[:nl+1] + shim + content[nl+1:]
    open('automl/AutoencoderKL.py', 'w').write(content)
    print('randn_tensor shim added')
else:
    print('randn_tensor already fixed or not found')
PYEOF

echo "=== Testing all imports ==="
python3 - << 'PYEOF'
import sys, os
sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')
for name, mod in [
    ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition'),
    ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset'),
    ('MVDiffusionImagePipeline', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image'),
    ('AutoencoderKL', 'automl.AutoencoderKL'),
]:
    try:
        m = __import__(mod, fromlist=[name])
        getattr(m, name)
        print(f'[PASS] {name}')
    except Exception as e:
        print(f'[FAIL] {name}: {str(e)[:100]}')
PYEOF

In [ ]:
%%bash
# COMPREHENSIVE SETUP: Clone repo, install deps, patch, and run CADDreamer
set -e

echo "=== Step 1: Clone CADDreamer repo ==="
if [ ! -d /content/CADDreamer ]; then
  cd /content && git clone https://github.com/lidan233/CADDreamer.git --quiet
    echo "Cloned OK"
    else
      echo "Already exists"
      fi

      echo "=== Step 2: Install compatible dependencies ==="
      pip install -q "diffusers==0.29.2" "huggingface_hub>=0.23.0" 2>/dev/null
      pip install -q "transformers>=4.35.0,<5.0.0" 2>/dev/null
      pip install -q einops omegaconf rembg onnxruntime segment-anything trimesh meshio 2>/dev/null
      pip install -q open3d 2>/dev/null || true
      echo "Deps installed"

      echo "=== Step 3: Apply all patches ==="
      cd /content/CADDreamer

      python3 - << 'PYEOF'
      import re, os

      # --- Patch 1: unet_mv2d_condition.py ---
      f = 'mvdiffusion/models/unet_mv2d_condition.py'
      c = open(f).read()

      # Fix unet_2d_blocks import path
      c = c.replace('from diffusers.models.unet_2d_blocks import', 'from diffusers.models.unets.unet_2d_blocks import')

      # Fix _load_state_dict_into_model
      if '_load_state_dict_into_model' in c:
          c = c.replace(
                  'from diffusers.models.modeling_utils import ModelMixin, load_state_dict, _load_state_dict_into_model',
                          'from diffusers.models.modeling_utils import ModelMixin, load_state_dict\ntry:\n    from diffusers.models.modeling_utils import _load_state_dict_into_model\nexcept ImportError:\n    def _load_state_dict_into_model(model, state_dict):\n        model.load_state_dict(state_dict, strict=False)\n        return [], [], [], []\n'
                              )

                              # Fix DIFFUSERS_CACHE
                              if 'DIFFUSERS_CACHE' in c and 'except ImportError' not in c[:2000]:
                                  shim = '\ntry:\n    from diffusers.utils import DIFFUSERS_CACHE\nexcept ImportError:\n    DIFFUSERS_CACHE = os.path.expanduser("~/.cache/huggingface/diffusers")\n'
                                      idx = c.find('import os\n')
                                          if idx < 0:
                                                  idx = c.find('from diffusers')
                                                          nl = c.find('\n', idx)
                                                                  c = c[:nl+1] + 'import os\n' + shim + c[nl+1:]
                                                                      else:
                                                                              c = c[:idx+len('import os\n')] + shim + c[idx+len('import os\n'):]

                                                                              open(f, 'w').write(c)
                                                                              print('Patch 1 (unet_mv2d_condition): OK')

                                                                              # --- Patch 2: AutoencoderKL.py ---
                                                                              f = 'automl/AutoencoderKL.py'
                                                                              c = open(f).read()

                                                                              # Fix vae imports
                                                                              c = c.replace('from diffusers.models.vae import', 'from diffusers.models.autoencoders.autoencoder_kl import')

                                                                              # Fix unet_2d_blocks
                                                                              c = c.replace('from diffusers.models.unet_2d_blocks import', 'from diffusers.models.unets.unet_2d_blocks import')

                                                                              # Fix FromOriginalVAEMixin
                                                                              c = c.replace('from diffusers.loaders import FromOriginalVAEMixin', '# from diffusers.loaders import FromOriginalVAEMixin\nFromOriginalVAEMixin = object')

                                                                              # Fix apply_forward_hook
                                                                              c = c.replace('from diffusers.utils import BaseOutput, apply_forward_hook',
                                                                                             'from diffusers.utils import BaseOutput\ntry:\n    from diffusers.utils.accelerate_utils import apply_forward_hook\nexcept ImportError:\n    apply_forward_hook = lambda fn: fn\n')

                                                                                             # Fix randn_tensor
                                                                                             if 'randn_tensor' in c and 'torch_utils' not in c:
                                                                                                 shim = '\ntry:\n    from diffusers.utils.torch_utils import randn_tensor\nexcept ImportError:\n    import torch as _torch\n    def randn_tensor(shape, generator=None, device=None, dtype=None):\n        return _torch.randn(shape, generator=generator, device=device, dtype=dtype)\n'
                                                                                                     idx = c.find('from diffusers')
                                                                                                         nl = c.find('\n', idx)
                                                                                                             c = c[:nl+1] + shim + c[nl+1:]

                                                                                                             open(f, 'w').write(c)
                                                                                                             print('Patch 2 (AutoencoderKL): OK')

                                                                                                             # --- Patch 3: pipeline_mvdiffusion_image.py ---
                                                                                                             f = 'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py'
                                                                                                             c = open(f).read()

                                                                                                             # Fix randn_tensor import
                                                                                                             c = re.sub(r'from diffusers\.utils import([^\n]*?)randn_tensor([^\n]*)\n',
                                                                                                                        'from diffusers.utils.torch_utils import randn_tensor\n', c)

                                                                                                                        # Fix FLAX_WEIGHTS_NAME
                                                                                                                        c = re.sub(r'from transformers\.utils import[^\n]*FLAX_WEIGHTS_NAME[^\n]*\n',
                                                                                                                                   'FLAX_WEIGHTS_NAME = "flax_model.msgpack"\n', c)

                                                                                                                                   open(f, 'w').write(c)
                                                                                                                                   print('Patch 3 (pipeline): OK')

                                                                                                                                   # --- Patch 4: unet_mv2d_blocks.py ---
                                                                                                                                   f = 'mvdiffusion/models/unet_mv2d_blocks.py'
                                                                                                                                   if os.path.exists(f):
                                                                                                                                       c = open(f).read()
                                                                                                                                           c = c.replace('from diffusers.models.unet_2d_blocks import', 'from diffusers.models.unets.unet_2d_blocks import')
                                                                                                                                               open(f, 'w').write(c)
                                                                                                                                                   print('Patch 4 (unet_mv2d_blocks): OK')

                                                                                                                                                   print('All patches applied!')
                                                                                                                                                   PYEOF

                                                                                                                                                   echo "=== Step 4: Verify all imports ==="
                                                                                                                                                   python3 - << 'PYEOF'
                                                                                                                                                   import sys, os
                                                                                                                                                   sys.path.insert(0, '/content/CADDreamer')
                                                                                                                                                   os.chdir('/content/CADDreamer')

                                                                                                                                                   tests = [
                                                                                                                                                       ('torch', 'torch'),
                                                                                                                                                           ('diffusers', 'diffusers'),
                                                                                                                                                               ('transformers', 'transformers'),
                                                                                                                                                                   ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition'),
                                                                                                                                                                       ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset'),
                                                                                                                                                                           ('MVDiffusionImagePipeline', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image'),
                                                                                                                                                                               ('AutoencoderKL', 'automl.AutoencoderKL'),
                                                                                                                                                                                   ('save_cache', 'utils.util'),
                                                                                                                                                                                   ]

                                                                                                                                                                                   all_pass = True
                                                                                                                                                                                   for name, mod in tests:
                                                                                                                                                                                       try:
                                                                                                                                                                                               m = __import__(mod, fromlist=[name])
                                                                                                                                                                                                       ver = getattr(m, '__version__', '') if mod == name else ''
                                                                                                                                                                                                               print(f'[PASS] {name}' + (f' {ver}' if ver else ''))
                                                                                                                                                                                                                   except Exception as e:
                                                                                                                                                                                                                           print(f'[FAIL] {name}: {str(e)[:100]}')
                                                                                                                                                                                                                                   all_pass = False

                                                                                                                                                                                                                                   if all_pass:
                                                                                                                                                                                                                                       print('\n*** ALL IMPORTS PASSED! Ready for inference. ***')
                                                                                                                                                                                                                                       else:
                                                                                                                                                                                                                                           print('\n*** Some imports failed - check errors above ***')
                                                                                                                                                                                                                                           PYEOF

In [ ]:
%%bash
# Fix numpy and PIL compatibility issues caused by open3d/rembg installing wrong versions
echo "=== Current numpy version ==="
python3 -c "import numpy; print('numpy:', numpy.__version__)"
python3 -c "import PIL; print('PIL:', PIL.__version__)"

echo "=== Reinstalling compatible numpy and pillow ==="
# open3d needs numpy>=2.0, but other packages need numpy<2
# Check what's installed
pip show numpy 2>/dev/null | grep Version
pip show pillow 2>/dev/null | grep Version

# The issue is numpy._core.umath._center doesn't exist - this suggests numpy 1.x was installed
# but code expects numpy 2.x, OR numpy 2.x was installed but something cached numpy 1.x
# Let's install a compatible numpy
pip install -q "numpy>=1.24,<2.0" 2>/dev/null || true

# For PIL._typing issue, install compatible pillow
pip install -q "pillow>=9.0,<11.0" 2>/dev/null || true

echo "=== After fix ==="
python3 -c "import numpy; print('numpy:', numpy.__version__)"
python3 -c "import PIL; print('PIL:', PIL.__version__)"

In [ ]:
%%bash
# Fix numpy and PIL compatibility issues
echo "=== Current versions ==="
python3 -c "import numpy; print('numpy:', numpy.__version__)"
python3 -c "import PIL; print('PIL:', PIL.__version__)"

echo "=== Checking what package requires new numpy ==="
python3 -c "
import pkg_resources
for pkg in ['open3d', 'trimesh', 'meshio', 'rembg']:
    try:
        v = pkg_resources.get_distribution(pkg).version
        print(pkg, v)
    except:
        print(pkg, 'not installed')
"

echo "=== Checking actual error for _center ==="
python3 -c "from numpy._core.umath import _center; print('_center OK')" 2>&1 || true
python3 -c "import numpy; print(dir(numpy._core.umath)[:5])" 2>&1 || true

echo "=== Reinstalling compatible versions ==="
# Install numpy 1.x (more compatible)
pip install -q "numpy>=1.24.0,<2.0.0" 2>/dev/null || true
# Install pillow < 10 to avoid _Ink issue
pip install -q "pillow>=9.0.0,<10.0.0" 2>/dev/null || true
echo "Done reinstall"

echo "=== Verify after fix ==="
python3 -c "import numpy; print('numpy:', numpy.__version__)"
python3 -c "import PIL; print('PIL:', PIL.__version__)"

echo "=== Test imports again ==="
python3 -c "
import sys, os
sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')
for name, mod in [
    ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition'),
    ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset'),
    ('AutoencoderKL', 'automl.AutoencoderKL'),
]:
    try:
        m = __import__(mod, fromlist=[name])
        getattr(m, name)
        print(f'[PASS] {name}')
    except Exception as e:
        print(f'[FAIL] {name}: {str(e)[:80]}')
"

In [ ]:
%%bash
# Fresh subprocess test - verify all imports work with new numpy/PIL
cd /content/CADDreamer
echo "=== Version Check ==="
python3 -c "import numpy; import PIL; print('numpy:', numpy.__version__, '| PIL:', PIL.__version__)"

echo "=== Full Import Test ==="
python3 -c "
import sys, os
sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')

passed = 0
for name, mod in [
    ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition'),
    ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset'),
    ('MVDiffusionImagePipeline', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image'),
    ('AutoencoderKL', 'automl.AutoencoderKL'),
    ('save_cache', 'utils.util'),
]:
    try:
        m = __import__(mod, fromlist=[name])
        getattr(m, name)
        print('[PASS]', name)
        passed += 1
    except Exception as e:
        print('[FAIL]', name + ':', str(e)[:100])
print('Score:', passed, '/5')
if passed == 5:
    print('ALL IMPORTS PASSED!')
"

In [ ]:
%%bash
cd /content/CADDreamer

echo "=== Fix 1: dual_transformer_2d module ==="
python3 -c "
import diffusers.models as m
attrs = [x for x in dir(m) if 'transformer' in x.lower() or 'dual' in x.lower()]
print('Transformer-related:', attrs[:10])
"
# Check where DualTransformer2DModel is
python3 -c "
try:
    from diffusers.models.transformers.dual_transformer_2d import DualTransformer2DModel
    print('dual_transformer in transformers submodule')
except:
    pass
try:
    from diffusers.models import DualTransformer2DModel
    print('dual_transformer in models')
except:
    pass
"
grep -n "dual_transformer_2d" mvdiffusion/models/unet_mv2d_blocks.py | head -5
grep -n "dual_transformer_2d" mvdiffusion/models/unet_mv2d_condition.py | head -5

echo "=== Fix pipeline logging ==="
# Replace getLogger with get_logger (diffusers logging style)
sed -i 's/diffusers.utils import logging/diffusers.utils import logging as dlogging/' mvdiffusion/pipelines/pipeline_mvdiffusion_image.py
sed -i 's/logging.getLogger/dlogging.get_logger/g' mvdiffusion/pipelines/pipeline_mvdiffusion_image.py
echo "Pipeline logging fixed"
grep -n "getLogger\|get_logger\|dlogging" mvdiffusion/pipelines/pipeline_mvdiffusion_image.py | head -5

echo "=== Fix SingleImageDataset CuPy ==="
head -30 mvdiffusion/data/single_image_dataset.py | head -30

echo "=== Apply dual_transformer_2d fix ==="
for f in mvdiffusion/models/unet_mv2d_blocks.py mvdiffusion/models/unet_mv2d_condition.py; do
    if grep -q "dual_transformer_2d" "$f" 2>/dev/null; then
        # Fix: dual_transformer_2d moved to transformers submodule
        sed -i 's|from diffusers.models.dual_transformer_2d import|from diffusers.models.transformers.dual_transformer_2d import|g' "$f"
        echo "Fixed: $f"
    fi
done

In [ ]:
%%bash
cd /content/CADDreamer

echo "=== unet_mv2d_condition.py lines 15-25 ==="
sed -n '15,25p' mvdiffusion/models/unet_mv2d_condition.py

echo "=== AutoencoderKL.py lines 35-45 ==="
sed -n '35,45p' automl/AutoencoderKL.py

echo "=== pipeline_mvdiffusion_image.py lines 25-40 ==="
sed -n '25,40p' mvdiffusion/pipelines/pipeline_mvdiffusion_image.py

echo "=== Fix AutoencoderKL.py - remove randn_tensor from the old import ==="
# The old import is: from diffusers.utils import BaseOutput, is_torch_version, randn_tensor
# Fix: change it to not include randn_tensor
sed -i 's/from diffusers.utils import BaseOutput, is_torch_version, randn_tensor/from diffusers.utils import BaseOutput, is_torch_version/' automl/AutoencoderKL.py
echo "AutoencoderKL fix applied"

echo "=== Fix pipeline - restore logging import ==="
# Line 35 has # logger = logging... which means logging import was removed
# Need to add: import logging
sed -i '35i import logging' mvdiffusion/pipelines/pipeline_mvdiffusion_image.py 2>/dev/null || true
# Or better - let's check what's at line 30
sed -n '28,38p' mvdiffusion/pipelines/pipeline_mvdiffusion_image.py

echo "=== Verify fixes ==="
python3 -c "
import sys, os
sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')
for name, mod in [
    ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition'),
    ('AutoencoderKL', 'automl.AutoencoderKL'),
    ('MVDiffusionImagePipeline', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image'),
    ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset'),
    ('save_cache', 'utils.util'),
]:
    try:
        m = __import__(mod, fromlist=[name])
        getattr(m, name)
        print('[PASS]', name)
    except Exception as e:
        print('[FAIL]', name + ':', str(e)[:100])
"

In [ ]:
%%bash
cd /content/CADDreamer
echo "=== Diagnosing unet_mv2d_condition.py ==="
grep -n "DIFFUSERS_CACHE\|_load_state_dict_into_model" mvdiffusion/models/unet_mv2d_condition.py | head -10

echo "=== Diagnosing AutoencoderKL.py ==="
grep -n "randn_tensor\|apply_forward_hook" automl/AutoencoderKL.py | head -10

echo "=== Diagnosing pipeline_mvdiffusion_image.py ==="
grep -n "logging\|randn_tensor\|FLAX_WEIGHTS" mvdiffusion/pipelines/pipeline_mvdiffusion_image.py | head -10

echo "=== Checking if segment_anything is installed ==="
python3 -c "import segment_anything; print('segment_anything OK')" 2>&1 || true

## Continuation added by Codex

These cells continue from the last failing import-diagnostic state in the notebook. Run Step 21 first. If it reaches Score: 5/5, run Step 22 to perform stage-1 inference when the required CADDreamer checkpoint/input folders are present.


In [ ]:
%%bash
# Step 21: Continue from Claude handoff — repair remaining imports and test 5/5
# This cell is idempotent: rerun it after a Colab runtime restart if needed.

set +e
cd /content
if [ ! -d /content/CADDreamer ]; then
  git clone https://github.com/lidan233/CADDreamer.git
fi

cd /content/CADDreamer

echo "=== GPU / Python environment ==="
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true
python3 - <<'PY'
import sys, torch
print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
PY

echo "=== Install/repair Python packages ==="
pip install -q "diffusers==0.29.2" "huggingface_hub>=0.23.0" "transformers>=4.35.0" accelerate einops omegaconf rembg onnxruntime segment-anything trimesh meshio open3d scikit-image opencv-python dill
pip install -q "numpy==1.26.4" "pillow==9.5.0"

# The previous notebook state showed a CuPy binary import failure while importing SingleImageDataset.
# Colab T4 currently uses CUDA 12.x PyTorch builds, so cupy-cuda12x is the correct wheel family.
python3 - <<'PY'
import subprocess, sys
try:
    import cupy
    print("cupy already imports:", cupy.__version__)
except Exception as e:
    print("cupy import issue, installing cupy-cuda12x:", str(e).split("\n")[0])
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "cupy"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])
PY

cat > /tmp/caddreamer_patch_phase1.py <<'PY'
from pathlib import Path
import re

root = Path("/content/CADDreamer")

def patch(path, replacements=()):
    p = root / path
    s = p.read_text()
    old = s
    for a, b in replacements:
        s = s.replace(a, b)
    if s != old:
        p.write_text(s)
        print("patched:", path)
    else:
        print("already ok/no-op:", path)

# 1) UNet: diffusers 0.29.2 moved/removed several imports.
unet_shim = '''from typing import Any, Dict, List, Optional, Tuple, Union
import os

try:
    from diffusers.utils import DIFFUSERS_CACHE
except Exception:
    DIFFUSERS_CACHE = os.path.expanduser("~/.cache/huggingface/diffusers")
try:
    from diffusers.utils import HF_HUB_OFFLINE
except Exception:
    HF_HUB_OFFLINE = False
try:
    from diffusers.models.modeling_utils import _load_state_dict_into_model
except Exception:
    def _load_state_dict_into_model(model_to_load, state_dict):
        model_to_load.load_state_dict(state_dict, strict=False)
        return []
'''

p = root / "mvdiffusion/models/unet_mv2d_condition.py"
s = p.read_text()
s = re.sub(
    r"from typing import Any, Dict, List, Optional, Tuple, Union\s+import os\s+(try:\s+from diffusers\\.utils import DIFFUSERS_CACHE.*?\n\s*)?import torch",
    unet_shim + "\nimport torch",
    s,
    flags=re.S,
)
s = s.replace("from diffusers.models.modeling_utils import ModelMixin, load_state_dict, _load_state_dict_into_model", "from diffusers.models.modeling_utils import ModelMixin, load_state_dict")
s = s.replace("from diffusers.models.unet_2d_blocks import", "from diffusers.models.unets.unet_2d_blocks import")
s = s.replace("from diffusers.models.dual_transformer_2d import", "from diffusers.models.transformers.dual_transformer_2d import")
s = s.replace("    DIFFUSERS_CACHE,\n", "")
s = s.replace("    HF_HUB_OFFLINE,\n", "")
p.write_text(s)
print("patched: mvdiffusion/models/unet_mv2d_condition.py")

# 2) MV blocks: moved imports.
patch("mvdiffusion/models/unet_mv2d_blocks.py", [
    ("from diffusers.models.attention import AdaGroupNorm", "from diffusers.models.normalization import AdaGroupNorm"),
    ("from diffusers.models.dual_transformer_2d import", "from diffusers.models.transformers.dual_transformer_2d import"),
    ("from diffusers.models.unet_2d_blocks import", "from diffusers.models.unets.unet_2d_blocks import"),
])

# 3) Pipeline: randn_tensor moved; keep diffusers logging module for get_logger().
p = root / "mvdiffusion/pipelines/pipeline_mvdiffusion_image.py"
s = p.read_text()
s = s.replace("from diffusers.utils import deprecate, logging, randn_tensor", "from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor")
s = s.replace("from diffusers.utils import deprecate, logging as dlogging, randn_tensor", "from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor")
s = s.replace("dlogging.get_logger", "logging.get_logger")
s = s.replace("\nimport logging\n# logger = logging.get_logger", "\n# logger = logging.get_logger")
p.write_text(s)
print("patched: mvdiffusion/pipelines/pipeline_mvdiffusion_image.py")

# 4) AutoencoderKL: moved VAE/unet/randn/apply_forward_hook imports; removed FromOriginalVAEMixin.
p = root / "automl/AutoencoderKL.py"
s = p.read_text()
s = s.replace("from diffusers.loaders import FromOriginalVAEMixin", "try:\n    from diffusers.loaders import FromOriginalVAEMixin\nexcept Exception:\n    FromOriginalVAEMixin = object")
s = s.replace("from diffusers.utils import BaseOutput, apply_forward_hook", "from diffusers.utils import BaseOutput\ntry:\n    from diffusers.utils.accelerate_utils import apply_forward_hook\nexcept Exception:\n    apply_forward_hook = lambda fn: fn")
s = s.replace("from diffusers.models.vae import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder", "from diffusers.models.autoencoders.autoencoder_kl import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder")
s = s.replace("from diffusers.utils import BaseOutput, is_torch_version, randn_tensor", "from diffusers.utils import BaseOutput, is_torch_version\ntry:\n    from diffusers.utils.torch_utils import randn_tensor\nexcept Exception:\n    from diffusers.utils import randn_tensor")
s = s.replace("from diffusers.models.unet_2d_blocks import", "from diffusers.models.unets.unet_2d_blocks import")
p.write_text(s)
print("patched: automl/AutoencoderKL.py")
PY

python3 /tmp/caddreamer_patch_phase1.py

echo "=== Final 5-module import test (fresh subprocess) ==="
cat > /tmp/caddreamer_import_test.py <<'PY'
import os, sys, traceback
sys.path.insert(0, "/content/CADDreamer")
os.chdir("/content/CADDreamer")
tests = [
    ("UNetMV2DConditionModel", "mvdiffusion.models.unet_mv2d_condition", "UNetMV2DConditionModel"),
    ("SingleImageDataset", "mvdiffusion.data.single_image_dataset", "SingleImageDataset"),
    ("MVDiffusionImagePipeline", "mvdiffusion.pipelines.pipeline_mvdiffusion_image", "MVDiffusionImagePipeline"),
    ("AutoencoderKL", "automl.AutoencoderKL", "AutoencoderKL"),
    ("save_cache", "utils.util", "save_cache"),
]
passed = 0
for label, mod, attr in tests:
    try:
        m = __import__(mod, fromlist=[attr])
        getattr(m, attr)
        print(f"[PASS] {label}")
        passed += 1
    except Exception as e:
        print(f"[FAIL] {label}: {type(e).__name__}: {str(e)[:400]}")
        traceback.print_exc(limit=2)
print(f"Score: {passed}/5")
if passed != len(tests):
    raise SystemExit(1)
print("ALL CORE CADDREAMER IMPORTS PASSED — ready for stage-1 inference checks.")
PY
python3 /tmp/caddreamer_import_test.py


In [ ]:
%%bash
# Step 22: Run CADDreamer stage-1 inference if the required model/input assets are present.
# This follows the repo README command and avoids a cryptic failure when external Box checkpoints
# have not yet been uploaded/extracted into the Colab runtime.

set +e
cd /content/CADDreamer

echo "=== Asset check ==="
missing=0
while read -r f; do
  if [ -e "$f" ]; then
    echo "[OK] $f"
  else
    echo "[MISSING] $f"
    missing=1
  fi
done <<'EOF'
ckpts/wonder3d-v1.0/model_index.json
ckpts/wonder3d-v1.0/unet/diffusion_pytorch_model.bin
ckpts/wonder3d-v1.0/vae/diffusion_pytorch_model.bin
finetuning_vae_normal/outputs/finetune_vae/vae_16.pth
finetuning_vae_normal/outputs/finetune_vae/mlp_16.pth
finetuning_vae_normal/outputs/finetuning_vae_normal/vae_normal_2.pth
our_inputs/test_real_images/testnormal_0_0.png
EOF

if [ "$missing" -ne 0 ]; then
  echo ""
  echo "Imports are the Phase-1 code milestone. Full generation needs the external assets from the README:"
  echo "  - ckpts/wonder3d-v1.0"
  echo "  - finetuning_vae_normal/outputs/..."
  echo "  - input example under our_inputs/test_real_images"
  echo "Upload/extract those folders into /content/CADDreamer, then rerun this cell."
  exit 0
fi

echo "=== Running CADDreamer stage-1 multi-view generation + NEUS mesh step ==="
python3 test_mvdiffusion_seq.py   --config configs/train/testing_4090_stage_1_cad_6views-lvis.yaml   --idx 0   --gpu 0

echo "=== Generated files ==="
find test_outputs cached_output -maxdepth 4 -type f 2>/dev/null | head -80


In [ ]:
%%bash

cd /content/CADDreamer

python3 -c "import torch,diffusers,transformers,huggingface_hub,numpy,PIL; print('torch',torch.__version__); print('diffusers',diffusers.__version__); print('transformers',transformers.__version__); print('hub',huggingface_hub.__version__); print('numpy',numpy.__version__); print('PIL',PIL.__version__)"

python3 /tmp/caddreamer_import_test.py 2>&1 || true

In [ ]:
%%bash
cd /content

unzip -q "finetuning_vae_normal (1)" -d /content/CADDreamer
unzip -q "ckpts (2)" -d /content/CADDreamer

%%bash
cd /content/CADDreamer

for f in \
  ckpts/wonder3d-v1.0/model_index.json \
  ckpts/wonder3d-v1.0/unet/diffusion_pytorch_model.bin \
  ckpts/wonder3d-v1.0/vae/diffusion_pytorch_model.bin \
  finetuning_vae_normal/outputs/finetune_vae/vae_16.pth \
  finetuning_vae_normal/outputs/finetune_vae/mlp_16.pth \
  finetuning_vae_normal/outputs/finetuning_vae_normal/vae_normal_2.pth
do
  if [ -e "$f" ]; then
    echo "[OK] $f"
  else
    echo "[MISSING] $f"
  fi
done

In [ ]:
import os
import shutil

# Source and destination paths
src = '/finetuning_vae_normal (1).zip'
dst_folder = '/content/CADDreamer'
dst_zip = '/content/finetuning_vae_normal.zip'

os.makedirs(dst_folder, exist_ok=True)

print('=== Moving and Extracting Assets ===')
try:
    # Move from root to /content for easier handling
    if os.path.exists(src):
        shutil.move(src, dst_zip)
        print(f'Moved {src} to {dst_zip}')

    # Unzip into the repo directory
    import zipfile
    with zipfile.ZipFile(dst_zip, 'r') as zip_ref:
        zip_ref.extractall(dst_folder)
    print(f'Extracted {dst_zip} to {dst_folder}')

    # Verify specific expected paths
    check_paths = [
        'finetuning_vae_normal/outputs/finetune_vae/vae_16.pth',
        'finetuning_vae_normal/outputs/finetuning_vae_normal/vae_normal_2.pth'
    ]
    for p in check_paths:
        full_p = os.path.join(dst_folder, p)
        print(f"[CHECK] {p}: {'EXISTS' if os.path.exists(full_p) else 'MISSING'}")

except Exception as e:
    print(f'Error during asset preparation: {e}')

In [ ]:
import os

print('=== Checking Root for Uploaded Checkpoints ===')
# The user previously tried to unzip 'ckpts (2)', so we check for variations in root
root_files = os.listdir('/')
candidate_zips = [f for f in root_files if 'ckpts' in f.lower() or 'wonder' in f.lower()]

if candidate_zips:
    print('Found potential checkpoint files in root:')
    for f in candidate_zips:
        print(f' - /{f}')
else:
    print('No checkpoint files found in root. Please ensure the \'ckpts (2).zip\' file is uploaded to the root or /content directory.')

In [ ]:
import os

print('=== Checking /content for Uploaded Checkpoints ===')
content_files = os.listdir('/content')
candidate_zips = [f for f in content_files if 'zip' in f.lower()]

if candidate_zips:
    print('Found the following ZIP files in /content:')
    for f in candidate_zips:
        print(f' - {f}')
else:
    print('No ZIP files found in /content.')

# Also check if it was already unzipped to a different folder name
subdirs = [d for d in content_files if os.path.isdir(os.path.join('/content', d))]
print(f'\nSubdirectories in /content: {subdirs}')

In [ ]:
import os

# Final check of other potential landing spots for uploaded files
search_paths = ['/tmp', '/content/CADDreamer']

print('=== Final Check for Checkpoints ===')
for path in search_paths:
    if os.path.exists(path):
        found = [f for f in os.listdir(path) if 'ckpts' in f.lower() or 'zip' in f.lower()]
        print(f'In {path}: {found if found else "None found"}')

# Check if there are any hidden folders in CADDreamer that might contain the data
if os.path.exists('/content/CADDreamer'):
    print('\nChecking CADDreamer subdirectories for existing ckpts folder:')
    for root, dirs, files in os.walk('/content/CADDreamer'):
        if 'ckpts' in dirs:
            print(f'Found ckpts folder at: {os.path.join(root, "ckpts")}')

In [ ]:
import os
import time
from IPython.display import clear_output

# Path to watch for
target_file = '/content/ckpts (2).zip'

print('Waiting for ckpts (2).zip to appear...')
try:
    while not os.path.exists(target_file):
        time.sleep(5)
        # Optional: Print a dot every 5 seconds to show it's running
        print('.', end='', flush=True)

    clear_output()
    file_size = os.path.getsize(target_file) / (1024 * 1024)
    print(f'✅ Found {target_file} ({file_size:.2f} MB)')
    print('The file is now available. We can proceed with extraction!')
except KeyboardInterrupt:
    print('\nStopped monitoring.')

### Option: Use Google Drive for Large File Transfer
1. Upload `ckpts (2).zip` to your Google Drive.
2. Run the cell below to mount your drive.
3. Run the following cell to copy the file into the environment.

In [ ]:
from pathlib import Path
for p in sorted(Path('/content').glob('*.zip')):
    try:
        print(p.name, 'exists=', p.exists(), 'size=', p.stat().st_size)
    except Exception as e:
        print(p.name, 'ERROR', e)

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive', force_remount=True)
src = Path('/content/drive/MyDrive/ckpts (1).zip')
dst = Path('/content/ckpts (1).zip')
print('src exists =', src.exists())
if src.exists():
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
    print('dst exists =', dst.exists(), 'size =', dst.stat().st_size)
else:
    print('Drive file missing:', src)

In [ ]:
from pathlib import Path
root = Path('/content/CADDreamer')
print('Top level:')
for p in sorted(root.iterdir()):
    print(('DIR ' if p.is_dir() else 'FILE'), p.name)
print('cadlib exists =', (root/'cadlib').exists())
print('DeepCAD exists =', (root/'DeepCAD').exists())
if (root/'DeepCAD').exists():
    for p in sorted((root/'DeepCAD').iterdir()):
        print('DEEPCAD', ('DIR ' if p.is_dir() else 'FILE'), p.name)
print('SHIM CELL TARGETED')from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
# After mounting, this checks for the specific checkpoint files in your Drive
drive_path = '/content/drive/MyDrive/ckpts (1).zip'
if os.path.exists(drive_path):
    print(f'Found: {drive_path}')
else:
    print('File not found. If you shared it from another account, ensure you have added a shortcut to it in your own "My Drive".')

In [ ]:
from pathlib import Path
root = Path('/content/drive')
print('drive exists =', root.exists())
for base in [root, root/'MyDrive']:
    print('BASE', base, 'exists=', base.exists())
    if base.exists():
        hits = []
        for p in base.rglob('*'):
            name = p.name.lower()
            if any(k in name for k in ['ckpts', 'image2csg', 'finetuning_vae_normal']):
                hits.append(str(p))
            if len(hits) >= 30:
                break
        for h in hits:
            print(h)
        print('shown=', len(hits))


In [ ]:
from google.colab import drive
import os
import zipfile
import shutil

# 1. Mount Drive
drive.mount('/content/drive', force_remount=True)

# 2. Define Paths
src_zip = '/content/drive/MyDrive/caddreamer-ckpts/ckpts (1).zip'
dst_repo = '/content/CADDreamer'
dst_ckpt_folder = os.path.join(dst_repo, 'ckpts')

# 3. Verify and Extract
if os.path.exists(src_zip):
    print(f'✅ Found zip at {src_zip}')
    os.makedirs(dst_ckpt_folder, exist_ok=True)

    print('Extracting checkpoints...')
    with zipfile.ZipFile(src_zip, 'r') as zip_ref:
        zip_ref.extractall(dst_ckpt_folder)
    print(f'✅ Extracted to {dst_ckpt_folder}')

    # Verification of contents
    print('\nListing extracted ckpts contents:')
    display(os.listdir(dst_ckpt_folder))
else:
    print(f'❌ File NOT found at {src_zip}')
    print('Please check if the folder name "caddreamer-ckpts" is exactly correct.')

In [ ]:
print('DOM EDIT TEST')

In [ ]:
import importlib, traceback, sys
from pathlib import Path
sys.path.insert(0, '/content/CADDreamer')

tests = [
    ('mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    ('mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    ('mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    ('automl.AutoencoderKL', 'AutoencoderKL'),
    ('cadlib.file_utils', 'save_cache'),
]
score = 0
for mod, attr in tests:
    try:
        m = importlib.import_module(mod)
        getattr(m, attr)
        print('[PASS]', attr)
        score += 1
    except Exception as e:
        print('[FAIL]', attr, type(e).__name__ + ':', e)
        traceback.print_exc(limit=1)
print('Score:', f'{score}/{len(tests)}')


In [ ]:
from pathlib import Path
import subprocess, shutil, zipfile

root = Path('/content/CADDreamer')
repo_tmp = Path('/content/CADDreamer_repo')
ckpt_zip = Path('/content/drive/MyDrive/caddreamer-ckpts/ckpts (1).zip')
small_zips = [Path('/content/image2csg_inputs.zip'), Path('/content/finetuning_vae_normal (1).zip')]

print('root exists before =', root.exists())
if root.exists():
    shutil.rmtree(root)
root.mkdir(parents=True, exist_ok=True)

if repo_tmp.exists():
    shutil.rmtree(repo_tmp)
print('cloning repo...')
subprocess.check_call(['git', 'clone', 'https://github.com/lidan233/CADDreamer.git', str(repo_tmp)])
for item in repo_tmp.iterdir():
    target = root / item.name
    if item.is_dir():
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)
print('repo copied')

print('ckpt zip exists =', ckpt_zip.exists())
with zipfile.ZipFile(ckpt_zip, 'r') as zf:
    (root / 'ckpts').mkdir(exist_ok=True)
    zf.extractall(root / 'ckpts')
print('ckpts extracted')
inner = root / 'ckpts' / 'ckpts'
if inner.exists():
    for child in list(inner.iterdir()):
        dst = root / 'ckpts' / child.name
        if not dst.exists():
            shutil.move(str(child), str(dst))
    try:
        inner.rmdir()
    except OSError:
        pass

for z in small_zips:
    print(z, 'exists=', z.exists())
    with zipfile.ZipFile(z, 'r') as zf:
        zf.extractall(root)
    print('extracted', z.name)

expected_png = root / 'our_inputs' / 'test_real_images' / 'testnormal_0_0.png'
if not expected_png.exists():
    cands = list(root.rglob('testnormal_0_0.png'))
    print('png candidates =', [str(c) for c in cands[:5]])
    if cands:
        expected_png.parent.mkdir(parents=True, exist_ok=True)
        if cands[0] != expected_png:
            shutil.copy2(cands[0], expected_png)

required = [
    root / 'test_mvdiffusion_seq.py',
    root / 'ckpts/wonder3d-v1.0/model_index.json',
    root / 'ckpts/wonder3d-v1.0/unet/diffusion_pytorch_model.bin',
    root / 'ckpts/wonder3d-v1.0/vae/diffusion_pytorch_model.bin',
    root / 'finetuning_vae_normal/outputs/finetune_vae/vae_16.pth',
    root / 'finetuning_vae_normal/outputs/finetune_vae/mlp_16.pth',
    root / 'finetuning_vae_normal/outputs/finetuning_vae_normal/vae_normal_2.pth',
    root / 'our_inputs/test_real_images/testnormal_0_0.png',
]
print('=== required ===')
for p in required:
    print('[OK]' if p.exists() else '[MISSING]', p)


In [ ]:
from pathlib import Path
root = Path('/content/drive')
print('drive exists =', root.exists())
if root.exists():
    names = ['finetuning_vae_normal', 'image2csg']
    hits = []
    for p in root.rglob('*'):
        name = p.name.lower()
        if any(k in name for k in names):
            hits.append(str(p))
            if len(hits) >= 50:
                break
    print('hits =', len(hits))
    for h in hits:
        print(h)


In [ ]:
from pathlib import Path
import json
checks = {
  'drive_exists': Path('/content/drive').exists(),
  'repo_exists': Path('/content/CADDreamer').exists(),
  'drive_ckpt': Path('/content/drive/MyDrive/caddreamer-ckpts/ckpts (1).zip').exists(),
  'drive_finetune': Path('/content/drive/MyDrive/finetuning_vae_normal (1).zip').exists(),
  'drive_inputs': Path('/content/drive/MyDrive/image2csg_inputs.zip').exists(),
  'content_finetune': Path('/content/finetuning_vae_normal (1).zip').exists(),
  'content_inputs': Path('/content/image2csg_inputs.zip').exists(),
}
print(json.dumps(checks, indent=2))
for p in [Path('/content/finetuning_vae_normal (1).zip'), Path('/content/image2csg_inputs.zip')]:
    if p.exists():
        print(p.name, 'size=', p.stat().st_size)
from google.colab import drive
drive.mount('/content/drive')
print('mounted attempt done')


In [ ]:
from pathlib import Path
import json
root = Path('/content/drive/MyDrive')
checks = {
  'drive_mounted': Path('/content/drive').exists(),
  'mydrive_exists': root.exists(),
  'ckpt_zip': (root / 'caddreamer-ckpts' / 'ckpts (1).zip').exists() or (root / 'ckpts (1).zip').exists(),
  'finetune_zip': (root / 'finetuning_vae_normal (1).zip').exists(),
  'inputs_zip': (root / 'image2csg_inputs.zip').exists(),
}
print(json.dumps(checks, indent=2))
if root.exists():
    hits=[]
    for p in root.rglob('*'):
        n=p.name.lower()
        if any(k in n for k in ['ckpts (1).zip','finetuning_vae_normal','image2csg_inputs']):
            hits.append(str(p))
            if len(hits) >= 20:
                break
    print('HITS')
    for h in hits:
        print(h)

In [ ]:
from pathlib import Path
import json
root = Path('/content/drive/MyDrive')
checks = {
  'drive_mounted': Path('/content/drive').exists(),
  'mydrive_exists': root.exists(),
  'ckpt_zip': (root / 'caddreamer-ckpts' / 'ckpts (1).zip').exists() or (root / 'ckpts (1).zip').exists(),
  'finetune_zip': (root / 'finetuning_vae_normal (1).zip').exists(),
  'inputs_zip': (root / 'image2csg_inputs.zip').exists(),
}
print(json.dumps(checks, indent=2))
if root.exists():
    hits = []
    for p in root.rglob('*'):
        n = p.name.lower()
        if any(k in n for k in ['ckpts (1).zip', 'finetuning_vae_normal', 'image2csg_inputs']):
            hits.append(str(p))
            if len(hits) >= 20:
                break
    print('HITS')
    for h in hits:
        print(h)


In [ ]:
from google.colab import drive
try:
    drive.flush_and_unmount()
    print('unmounted')
except Exception as e:
    print('unmount note', e)
drive.mount('/content/drive', force_remount=True)
print('remount done')

In [ ]:
from pathlib import Path
import json
root = Path('/content/drive/MyDrive')
checks = {
  'drive_mounted': Path('/content/drive').exists(),
  'mydrive_exists': root.exists(),
  'ckpt_zip': (root / 'caddreamer-ckpts' / 'ckpts (1).zip').exists() or (root / 'ckpts (1).zip').exists(),
  'finetune_zip': (root / 'finetuning_vae_normal (1).zip').exists(),
  'inputs_zip': (root / 'image2csg_inputs.zip').exists(),
}
print(json.dumps(checks, indent=2))
if root.exists():
    hits = []
    for p in root.rglob('*'):
        n = p.name.lower()
        if any(k in n for k in ['ckpts (1).zip', 'finetuning_vae_normal', 'image2csg_inputs']):
            hits.append(str(p))
            if len(hits) >= 20:
                break
    print('HITS')
    for h in hits:
        print(h)


In [ ]:
from pathlib import Path
zs = sorted(Path('/content').glob('*.zip'))
print('ZIPCOUNT', len(zs))
for p in zs:
    print(p.name, 'size=', p.stat().st_size)
from pathlib import Path
import shutil, zipfile, subprocess

def find_first(base: Path, patterns):
    for pat in patterns:
        hits = list(base.rglob(pat))
        if hits:
            return hits[0]
    return None

mydrive = Path('/content/drive/MyDrive')
ckpt_zip = find_first(mydrive, ['ckpts (1).zip'])
finetune_zip = find_first(mydrive, ['finetuning_vae_normal*.zip'])
inputs_zip = find_first(mydrive, ['image2csg_inputs.zip'])
print('ckpt_zip =', ckpt_zip)
print('finetune_zip =', finetune_zip)
print('inputs_zip =', inputs_zip)
for p in [ckpt_zip, finetune_zip, inputs_zip]:
    if p is None:
        raise FileNotFoundError('Missing required archive in Drive')
    print(p.name, 'size=', p.stat().st_size)

root = Path('/content/CADDreamer')
repo_tmp = Path('/content/CADDreamer_repo')
if root.exists():
    shutil.rmtree(root)
if repo_tmp.exists():
    shutil.rmtree(repo_tmp)
root.mkdir(parents=True, exist_ok=True)
print('cloning repo...')
subprocess.check_call(['git', 'clone', 'https://github.com/lidan233/CADDreamer.git', str(repo_tmp)])
for item in repo_tmp.iterdir():
    target = root / item.name
    if item.is_dir():
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)
print('repo copied')

(root / 'ckpts').mkdir(exist_ok=True)
with zipfile.ZipFile(ckpt_zip, 'r') as zf:
    zf.extractall(root / 'ckpts')
print('ckpts extracted')
inner = root / 'ckpts' / 'ckpts'
if inner.exists():
    for child in list(inner.iterdir()):
        dst = root / 'ckpts' / child.name
        if not dst.exists():
            shutil.move(str(child), str(dst))
    try:
        inner.rmdir()
    except OSError:
        pass

for z in [finetune_zip, inputs_zip]:
    with zipfile.ZipFile(z, 'r') as zf:
        zf.extractall(root)
    print('extracted', z.name)

expected_png = root / 'our_inputs' / 'test_real_images' / 'testnormal_0_0.png'
if not expected_png.exists():
    cands = list(root.rglob('testnormal_0_0.png'))
    print('png candidates =', [str(c) for c in cands[:5]])
    if cands:
        expected_png.parent.mkdir(parents=True, exist_ok=True)
        if cands[0] != expected_png:
            shutil.copy2(cands[0], expected_png)

required = [
    root / 'test_mvdiffusion_seq.py',
    root / 'ckpts/wonder3d-v1.0/model_index.json',
    root / 'ckpts/wonder3d-v1.0/unet/diffusion_pytorch_model.bin',
    root / 'ckpts/wonder3d-v1.0/vae/diffusion_pytorch_model.bin',
    root / 'finetuning_vae_normal/outputs/finetune_vae/vae_16.pth',
    root / 'finetuning_vae_normal/outputs/finetune_vae/mlp_16.pth',
    root / 'finetuning_vae_normal/outputs/finetuning_vae_normal/vae_normal_2.pth',
    root / 'our_inputs/test_real_images/testnormal_0_0.png',
]
print('=== required ===')
for p in required:
    print('[OK]' if p.exists() else '[MISSING]', p)


In [ ]:
from pathlib import Path
for p in sorted(Path('/content').glob('*.zip')):
    print(p.name, 'size=', p.stat().st_size)


In [ ]:
from pathlib import Path
import zipfile, json
root = Path('/content/drive/MyDrive')
patterns = ['ckpts*.zip', 'finetuning_vae_normal*.zip', 'image2csg_inputs*.zip']
print('CHECK_START')
print('drive_exists', Path('/content/drive').exists())
print('mydrive_exists', root.exists())
for pat in patterns:
    hits = sorted(root.rglob(pat)) if root.exists() else []
    print('PATTERN', pat, 'COUNT', len(hits))
    for p in hits[:10]:
        size = p.stat().st_size if p.exists() else None
        iszip = zipfile.is_zipfile(p) if p.is_file() else False
        print(str(p), 'size=', size, 'is_zip=', iszip)
print('CHECK_END')

In [ ]:
from pathlib import Path
import shutil, zipfile, subprocess

ckpt_zip = Path('/content/drive/MyDrive/caddreamer1/ckpts.zip')
finetune_zip = Path('/content/drive/MyDrive/caddreamer1/finetuning_vae_normal.zip')
inputs_zip = Path('/content/drive/MyDrive/caddreamer1/image2csg_inputs.zip')

for p in [ckpt_zip, finetune_zip, inputs_zip]:
    print('ZIP', p, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else None, 'is_zip=', zipfile.is_zipfile(p) if p.exists() else None)

root = Path('/content/CADDreamer')
repo_tmp = Path('/content/CADDreamer_repo')
if root.exists():
    shutil.rmtree(root)
if repo_tmp.exists():
    shutil.rmtree(repo_tmp)

root.mkdir(parents=True, exist_ok=True)
print('cloning repo...')
subprocess.check_call(['git', 'clone', 'https://github.com/lidan233/CADDreamer.git', str(repo_tmp)])
for item in repo_tmp.iterdir():
    target = root / item.name
    if item.is_dir():
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)
print('repo copied')

(root / 'ckpts').mkdir(exist_ok=True)
with zipfile.ZipFile(ckpt_zip, 'r') as zf:
    zf.extractall(root / 'ckpts')
print('ckpts extracted')
inner = root / 'ckpts' / 'ckpts'
if inner.exists():
    for child in list(inner.iterdir()):
        dst = root / 'ckpts' / child.name
        if not dst.exists():
            shutil.move(str(child), str(dst))
    try:
        inner.rmdir()
    except OSError:
        pass

with zipfile.ZipFile(finetune_zip, 'r') as zf:
    zf.extractall(root)
print('finetune extracted')
with zipfile.ZipFile(inputs_zip, 'r') as zf:
    zf.extractall(root)
print('inputs extracted')

expected_png = root / 'our_inputs' / 'test_real_images' / 'testnormal_0_0.png'
if not expected_png.exists():
    cands = list(root.rglob('testnormal_0_0.png'))
    print('png candidates =', [str(c) for c in cands[:5]])
    if cands:
        expected_png.parent.mkdir(parents=True, exist_ok=True)
        if cands[0] != expected_png:
            shutil.copy2(cands[0], expected_png)

required = [
    root / 'test_mvdiffusion_seq.py',
    root / 'ckpts/wonder3d-v1.0/model_index.json',
    root / 'ckpts/wonder3d-v1.0/unet/diffusion_pytorch_model.bin',
    root / 'ckpts/wonder3d-v1.0/vae/diffusion_pytorch_model.bin',
    root / 'finetuning_vae_normal/outputs/finetune_vae/vae_16.pth',
    root / 'finetuning_vae_normal/outputs/finetune_vae/mlp_16.pth',
    root / 'finetuning_vae_normal/outputs/finetuning_vae_normal/vae_normal_2.pth',
    root / 'our_inputs/test_real_images/testnormal_0_0.png',
]
print('=== required ===')
for p in required:
    print('[OK]' if p.exists() else '[MISSING]', p)
print('RESTORE_DONE')

In [ ]:
from pathlib import Path
import os, sys

root = Path('/content/CADDreamer')

# UNet condition model
p = root / 'mvdiffusion/models/unet_mv2d_condition.py'
s = p.read_text()
s = s.replace('from diffusers.models.modeling_utils import ModelMixin, load_state_dict, _load_state_dict_into_model',
              'from diffusers.models.modeling_utils import ModelMixin, load_state_dict')
s = s.replace('from diffusers.models.unet_2d_blocks import',
              'from diffusers.models.unets.unet_2d_blocks import')
s = s.replace(' DIFFUSERS_CACHE,\n', '').replace(' HF_HUB_OFFLINE,\n', '')
shim = '''\ntry:
    from diffusers.utils import DIFFUSERS_CACHE
except Exception:
    DIFFUSERS_CACHE = str(Path.home() / ".cache/huggingface/diffusers")
try:
    from diffusers.utils import HF_HUB_OFFLINE
except Exception:
    HF_HUB_OFFLINE = False
try:
    from diffusers.models.modeling_utils import _load_state_dict_into_model
except Exception:
    def _load_state_dict_into_model(model_to_load, state_dict):
        model_to_load.load_state_dict(state_dict, strict=False)
        return []
'''
if 'DIFFUSERS_CACHE = str(Path.home()' not in s:
    s = 'from pathlib import Path\n' + shim + s
p.write_text(s)

# MV blocks
p = root / 'mvdiffusion/models/unet_mv2d_blocks.py'
s = p.read_text()
s = s.replace('from diffusers.models.attention import AdaGroupNorm',
              'from diffusers.models.normalization import AdaGroupNorm')
s = s.replace('from diffusers.models.dual_transformer_2d import',
              'from diffusers.models.transformers.dual_transformer_2d import')
s = s.replace('from diffusers.models.unet_2d_blocks import',
              'from diffusers.models.unets.unet_2d_blocks import')
p.write_text(s)

# Transformer decorator
p = root / 'mvdiffusion/models/transformer_mv2d.py'
s = p.read_text()
s = s.replace(
    'from diffusers.utils import BaseOutput, deprecate, maybe_allow_in_graph',
    '''from diffusers.utils import BaseOutput, deprecate
try:
    from diffusers.utils.torch_utils import maybe_allow_in_graph
except ImportError:
    def maybe_allow_in_graph(fn):
        return fn'''
)
p.write_text(s)

# Pipeline
p = root / 'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py'
s = p.read_text()
s = s.replace('from diffusers.utils import deprecate, logging, randn_tensor',
              'from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor')
s = s.replace('from diffusers.utils import deprecate, logging as dlogging, randn_tensor',
              'from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor')
s = s.replace('dlogging.get_logger', 'logging.get_logger')
p.write_text(s)

# Autoencoder
p = root / 'automl/AutoencoderKL.py'
s = p.read_text()
s = s.replace('from diffusers.loaders import FromOriginalVAEMixin',
              'try:\n from diffusers.loaders import FromOriginalVAEMixin\nexcept Exception:\n FromOriginalVAEMixin = object')
s = s.replace('from diffusers.utils import BaseOutput, apply_forward_hook',
              'from diffusers.utils import BaseOutput\ntry:\n from diffusers.utils.accelerate_utils import apply_forward_hook\nexcept Exception:\n apply_forward_hook = lambda fn: fn')
s = s.replace('from diffusers.models.vae import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder',
              'from diffusers.models.autoencoders.autoencoder_kl import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder')
s = s.replace('from diffusers.utils import BaseOutput, is_torch_version, randn_tensor',
              'from diffusers.utils import BaseOutput, is_torch_version\nfrom diffusers.utils.torch_utils import randn_tensor')
s = s.replace('from diffusers.models.unet_2d_blocks import',
              'from diffusers.models.unets.unet_2d_blocks import')
p.write_text(s)

print('CADDreamer compatibility patches applied')

sys.path.insert(0, str(root))
os.chdir(root)
tests = [
    ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    ('MVDiffusionImagePipeline', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    ('AutoencoderKL', 'automl.AutoencoderKL', 'AutoencoderKL'),
    ('save_cache', 'utils.util', 'save_cache'),
]
n = 0
for label, mod, attr in tests:
    try:
        m = __import__(mod, fromlist=[attr])
        getattr(m, attr)
        print('[PASS]', label)
        n += 1
    except Exception as e:
        print('[FAIL]', label, type(e).__name__, str(e)[:300])
print(f'Score: {n}/5')
print('PATCH_TEST_DONE')

In [ ]:
%pip install -q segment-anything trimesh meshio open3d rembg onnxruntime scikit-image opencv-python dill
print('DEPS_REPAIRED')

In [ ]:
%pip install -q --no-deps --force-reinstall pillow==9.5.0

import os, sys, torch, diffusers, transformers, huggingface_hub, numpy, PIL
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('diffusers', diffusers.__version__)
print('transformers', transformers.__version__)
print('hub', huggingface_hub.__version__)
print('numpy', numpy.__version__)
print('PIL', PIL.__version__)

sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')
tests = [
    ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    ('MVDiffusionImagePipeline', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    ('AutoencoderKL', 'automl.AutoencoderKL', 'AutoencoderKL'),
    ('save_cache', 'utils.util', 'save_cache'),
]
n = 0
for label, mod, attr in tests:
    try:
        m = __import__(mod, fromlist=[attr])
        getattr(m, attr)
        print('[PASS]', label)
        n += 1
    except Exception as e:
        print('[FAIL]', label, type(e).__name__, str(e)[:300])
print(f'Score: {n}/5')
print('VERIFY_DONE')

In [ ]:
bbimport subprocess, torch
print('TORCH_VERSION', torch.__version__)
print('CUDA_AVAILABLE', torch.cuda.is_available())
print('TORCH_CUDA_VERSION', getattr(torch.version, 'cuda', None))
print('DEVICE_COUNT', torch.cuda.device_count() if torch.cuda.is_available() else 0)
res = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('NVIDIA_SMI_RC', res.returncode)
print(res.stdout)
print(res.stderr)
print('GPU_CHECK_DONE')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('DRIVE_MOUNTED')

In [ ]:
%%bash
set +e

# Step 1: Install support deps first (before pinning core)
pip install -q segment-anything trimesh meshio rembg onnxruntime scikit-image opencv-python dill 2>/dev/null

# Step 2: Force-pin the exact known-good core stack
pip install -q --no-deps --force-reinstall --no-cache-dir \
  "diffusers==0.29.2" \
  "transformers==4.44.0" \
  "huggingface_hub==0.24.7" \
  "tokenizers==0.19.1" \
  "accelerate==0.33.0" 2>/dev/null

# Step 3: Fix numpy and pillow
pip install -q --force-reinstall "numpy==1.26.4" 2>/dev/null
pip install -q --no-deps --force-reinstall "pillow==9.5.0" 2>/dev/null

# Step 4: Verify GPU
nvidia-smi | head -20 || echo "nvidia-smi FAILED"
python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

echo "DEPS_DONE"

In [ ]:
from pathlib import Path
import shutil, zipfile, subprocess

ckpt_zip = Path('/content/drive/MyDrive/caddreamer1/ckpts.zip')
finetune_zip = Path('/content/drive/MyDrive/caddreamer1/finetuning_vae_normal.zip')
inputs_zip = Path('/content/drive/MyDrive/caddreamer1/image2csg_inputs.zip')

for p in [ckpt_zip, finetune_zip, inputs_zip]:
    print('ZIP', p.name, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else None)

root = Path('/content/CADDreamer')
repo_tmp = Path('/content/CADDreamer_repo')
if root.exists(): shutil.rmtree(root)
if repo_tmp.exists(): shutil.rmtree(repo_tmp)

root.mkdir(parents=True, exist_ok=True)
print('Cloning repo...')
subprocess.check_call(['git', 'clone', 'https://github.com/lidan233/CADDreamer.git', str(repo_tmp)])
for item in repo_tmp.iterdir():
    target = root / item.name
    if item.is_dir(): shutil.copytree(item, target)
    else: shutil.copy2(item, target)
shutil.rmtree(repo_tmp)
print('Repo cloned to', root)

print('Extracting ckpts.zip...')
with zipfile.ZipFile(ckpt_zip) as z:
    z.extractall(root)

print('Extracting finetuning_vae_normal.zip...')
with zipfile.ZipFile(finetune_zip) as z:
    z.extractall(root)

print('Extracting image2csg_inputs.zip...')
with zipfile.ZipFile(inputs_zip) as z:
    z.extractall(root)

# Verify key files exist
checks = [
    root / 'test_mvdiffusion_seq.py',
    root / 'ckpts/wonder3d-v1.0/model_index.json',
    root / 'ckpts/wonder3d-v1.0/unet/diffusion_pytorch_model.bin',
    root / 'ckpts/wonder3d-v1.0/vae/diffusion_pytorch_model.bin',
    root / 'finetuning_vae_normal/outputs/finetune_vae/vae_16.pth',
    root / 'finetuning_vae_normal/outputs/finetune_vae/mlp_16.pth',
]
for c in checks:
    print('CHECK', c.name, c.exists())

# Check input images
import glob
imgs = glob.glob(str(root / 'our_inputs/**/*.png'), recursive=True)
print('Input images found:', len(imgs))
if imgs: print('  Sample:', imgs[:2])

print('RESTORE_DONE')

In [ ]:
import sys, os
from pathlib import Path

sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')
root = Path('/content/CADDreamer')

# ========== PATCH 1: unet_mv2d_condition.py ==========
p = root / 'mvdiffusion/models/unet_mv2d_condition.py'
s = p.read_text()

# Fix _load_state_dict_into_model - add stub
if '_load_state_dict_into_model' in s and 'def _load_state_dict_into_model' not in s:
    stub = """
try:
    from diffusers.models.modeling_utils import ModelMixin, load_state_dict, _load_state_dict_into_model
except ImportError:
    from diffusers.models.modeling_utils import ModelMixin, load_state_dict
    def _load_state_dict_into_model(model, state_dict, start_prefix=''):
        import torch
        model.load_state_dict({k[len(start_prefix):] if k.startswith(start_prefix) else k: v for k, v in state_dict.items()}, strict=False)
        return [], []
"""
    s = s.replace(
        'from diffusers.models.modeling_utils import ModelMixin, load_state_dict, _load_state_dict_into_model',
        stub.strip()
    )

# Fix unet_2d_blocks import path
s = s.replace(
    'from diffusers.models.unet_2d_blocks import',
    'from diffusers.models.unets.unet_2d_blocks import'
)
# Fix dual_transformer_2d import path
s = s.replace(
    'from diffusers.models.dual_transformer_2d import',
    'from diffusers.models.transformers.dual_transformer_2d import'
)
# Fix DIFFUSERS_CACHE and HF_HUB_OFFLINE
s = s.replace(
    'from diffusers.utils import DIFFUSERS_CACHE, HF_HUB_OFFLINE',
    'try:\n    from diffusers.utils import DIFFUSERS_CACHE\nexcept ImportError:\n    DIFFUSERS_CACHE = os.path.expanduser("~/.cache/huggingface/diffusers")\nHF_HUB_OFFLINE = False'
)
p.write_text(s)
print('PATCH unet_mv2d_condition.py: done')

# ========== PATCH 2: unet_mv2d_blocks.py ==========
p2 = root / 'mvdiffusion/models/unet_mv2d_blocks.py'
s2 = p2.read_text()
s2 = s2.replace(
    'from diffusers.models.unet_2d_blocks import',
    'from diffusers.models.unets.unet_2d_blocks import'
)
s2 = s2.replace(
    'from diffusers.models.dual_transformer_2d import',
    'from diffusers.models.transformers.dual_transformer_2d import'
)
# Fix AdaGroupNorm
s2 = s2.replace(
    'from diffusers.models.attention import AdaGroupNorm',
    'try:\n    from diffusers.models.attention import AdaGroupNorm\nexcept ImportError:\n    from diffusers.models.normalization import AdaGroupNorm'
)
p2.write_text(s2)
print('PATCH unet_mv2d_blocks.py: done')

# ========== PATCH 3: AutoencoderKL (automl/AutoencoderKL.py) ==========
# Find the file
vae_file = None
for candidate in [
    root / 'automl/AutoencoderKL.py',
    root / 'automl/autoencoder_kl.py',
]:
    if candidate.exists():
        vae_file = candidate
        break

if vae_file is None:
    import glob
    candidates = glob.glob(str(root / '**/*utoencoder*.py'), recursive=True)
    if candidates:
        vae_file = Path(candidates[0])

if vae_file:
    sv = vae_file.read_text()
    # Fix randn_tensor
    sv = sv.replace(
        'from diffusers.utils import randn_tensor',
        'try:\n    from diffusers.utils import randn_tensor\nexcept ImportError:\n    from diffusers.utils.torch_utils import randn_tensor'
    )
    # Fix apply_forward_hook
    sv = sv.replace(
        'from diffusers.utils import apply_forward_hook',
        'try:\n    from diffusers.utils import apply_forward_hook\nexcept ImportError:\n    try:\n        from diffusers.utils.accelerate_utils import apply_forward_hook\n    except ImportError:\n        def apply_forward_hook(fn): return fn'
    )
    # Fix FromOriginalVAEMixin
    sv = sv.replace(
        'from diffusers.loaders import FromOriginalVAEMixin',
        'try:\n    from diffusers.loaders import FromOriginalVAEMixin\nexcept ImportError:\n    class FromOriginalVAEMixin: pass'
    )
    # Fix vae module import
    sv = sv.replace(
        'from diffusers.models.vae import',
        'try:\n    from diffusers.models.vae import'
    )
    vae_file.write_text(sv)
    print('PATCH', vae_file.name, ': done')
else:
    print('WARNING: AutoencoderKL file not found')

# ========== PATCH 4: Pipeline ==========
pipe_file = root / 'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py'
if pipe_file.exists():
    sp = pipe_file.read_text()
    # Fix randn_tensor
    sp = sp.replace(
        'from diffusers.utils import randn_tensor',
        'try:\n    from diffusers.utils import randn_tensor\nexcept ImportError:\n    from diffusers.utils.torch_utils import randn_tensor'
    )
    # Fix pipeline logging
    sp = sp.replace(
        'import logging',
        'from diffusers.utils import logging as _diffusers_logging'
    )
    sp = sp.replace(
        'logging.getLogger(__name__)',
        '_diffusers_logging.get_logger(__name__)'
    )
    # Fix unet imports in pipeline too
    sp = sp.replace(
        'from diffusers.models.unet_2d_blocks import',
        'from diffusers.models.unets.unet_2d_blocks import'
    )
    pipe_file.write_text(sp)
    print('PATCH pipeline: done')
else:
    print('WARNING: pipeline file not found')

# ========== PATCH 5: SingleImageDataset - fix CuPy import ==========
ds_file = root / 'mvdiffusion/data/single_image_dataset.py'
if ds_file.exists():
    sd = ds_file.read_text()
    # Make CuPy optional
    sd = sd.replace(
        'import cupy as cp',
        'try:\n    import cupy as cp\nexcept ImportError:\n    cp = None'
    )
    ds_file.write_text(sd)
    print('PATCH single_image_dataset.py: done')
else:
    print('WARNING: single_image_dataset.py not found')

print('CADDreamer compatibility patches applied')

# Now run smoke test
tests = [
    ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    ('MVDiffusionImagePipeline', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    ('AutoencoderKL', 'automl.AutoencoderKL', 'AutoencoderKL'),
    ('save_cache', 'utils.util', 'save_cache'),
]
n = 0
for label, mod, attr in tests:
    try:
        m_mod = __import__(mod, fromlist=[attr])
        getattr(m_mod, attr)
        print('[PASS]', label)
        n += 1
    except Exception as e:
        print('[FAIL]', label, type(e).__name__, str(e)[:300])
print(f'Score: {n}/5')
print('SMOKE_TEST_DONE')

In [ ]:
# Fix 1: Install missing packages and fix versions in KERNEL
import subprocess, sys

# Install segment-anything
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segment-anything'], capture_output=True)

# Fix numpy to 1.26.4 (in kernel process)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'numpy==1.26.4'], capture_output=True)

# Fix PIL to 9.5.0 (in kernel process)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', 'pillow==9.5.0'], capture_output=True)

# Verify
import importlib
import importlib.util

for pkg in ['numpy', 'PIL', 'segment_anything']:
    spec = importlib.util.find_spec(pkg)
    if spec:
        # Reload to get fresh version
        try:
            mod = importlib.import_module(pkg)
            importlib.reload(mod)
            print(f'{pkg}: {getattr(mod, "__version__", "ok")}')
        except:
            print(f'{pkg}: installed but reload failed')
    else:
        print(f'{pkg}: still not found')
print('PKG_FIX_DONE')

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')
root = Path('/content/CADDreamer')

# ====== FIX 1: unet_mv2d_condition.py ======
# The file has lines 38-57 with broken imports. Let's read and do line-based fixes.
p = root / 'mvdiffusion/models/unet_mv2d_condition.py'
lines = p.read_text().split('\n')

# Find and fix line 38 (the bad import)
for i, line in enumerate(lines):
    if '_load_state_dict_into_model' in line and 'from diffusers.models.modeling_utils import' in line:
        # Replace with version that doesn't import _load_state_dict_into_model
        lines[i] = line.replace(', _load_state_dict_into_model', '')
        print(f'Fixed unet_cond line {i}: removed _load_state_dict_into_model from import')
        break

# Fix DIFFUSERS_CACHE - lines 55, 57 area - they are probably in a multi-line import
# Find the block
for i, line in enumerate(lines):
    if 'DIFFUSERS_CACHE,' in line.strip() or ('DIFFUSERS_CACHE' in line and 'HF_HUB_OFFLINE' not in line and 'from diffusers' not in line):
        # This is inside a multi-line import block - we need to handle this differently
        # Let's check what comes before
        print(f'Found DIFFUSERS_CACHE at line {i}: {repr(line[:100])}')
    if 'HF_HUB_OFFLINE,' in line.strip():
        print(f'Found HF_HUB_OFFLINE at line {i}: {repr(line[:100])}')

p.write_text('\n'.join(lines))

# ====== FIX 2: AutoencoderKL.py ======
# Read the file and show lines around the syntax error (line 29)
vae_file = root / 'automl/AutoencoderKL.py'
vae_lines = vae_file.read_text().split('\n')
print("VAE lines 25-35:")
for i in range(25, min(36, len(vae_lines))):
    print(f"  {i}: {repr(vae_lines[i][:120])}")
print('DIAG2_DONE')

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')
root = Path('/content/CADDreamer')

# ====== FIX AutoencoderKL.py ======
# Problem: orphaned try: at line 26, unet_2d_blocks old path at line 29
vae_file = root / 'automl/AutoencoderKL.py'
vae_lines = vae_file.read_text().split('\n')

for i, line in enumerate(vae_lines):
    # Fix orphaned try: block - replace with proper try/except
    if line.strip() == 'try:' and i+1 < len(vae_lines) and 'from diffusers.models.vae import' in vae_lines[i+1]:
        vae_lines[i] = 'try:'
        # line i+1 already has the import
        # Add except after the import line
        vae_lines.insert(i+2, 'except ImportError:')
        vae_lines.insert(i+3, '    pass  # vae module not available, will use local definitions')
        print(f'Fixed orphaned try: at line {i}')
        break

# Reload after modification
vae_lines2 = vae_lines  # already modified above
for i, line in enumerate(vae_lines2):
    # Fix unet_2d_blocks old path
    if 'from diffusers.models.unet_2d_blocks import' in line:
        vae_lines2[i] = line.replace('from diffusers.models.unet_2d_blocks import',
                                      'from diffusers.models.unets.unet_2d_blocks import')
        print(f'Fixed vae unet_2d_blocks at line {i}')
    # Fix FromOriginalVAEMixin
    if 'from diffusers.loaders import FromOriginalVAEMixin' in line:
        vae_lines2[i] = 'try:\n    from diffusers.loaders import FromOriginalVAEMixin\nexcept ImportError:\n    class FromOriginalVAEMixin: pass'
        print(f'Fixed vae FromOriginalVAEMixin at line {i}')
    # Fix apply_forward_hook
    if 'from diffusers.utils import apply_forward_hook' in line:
        vae_lines2[i] = 'try:\n    from diffusers.utils import apply_forward_hook\nexcept ImportError:\n    try:\n        from diffusers.utils.accelerate_utils import apply_forward_hook\n    except ImportError:\n        def apply_forward_hook(fn): return fn'
        print(f'Fixed vae apply_forward_hook at line {i}')

vae_file.write_text('\n'.join(vae_lines2))
print('FIXED AutoencoderKL.py')

# ====== FIX unet_mv2d_condition.py - DIFFUSERS_CACHE multi-line import ======
p = root / 'mvdiffusion/models/unet_mv2d_condition.py'
lines = p.read_text().split('\n')

# Find the multi-line import block containing DIFFUSERS_CACHE and HF_HUB_OFFLINE
# Need to look at context around lines 55, 57
print("Lines 50-65 of unet_mv2d_condition.py:")
for i in range(50, min(66, len(lines))):
    print(f"  {i}: {repr(lines[i][:100])}")

print('FIX_DIAG3_DONE')

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')
root = Path('/content/CADDreamer')

# ====== MASTER FIX: unet_mv2d_condition.py ======
p = root / 'mvdiffusion/models/unet_mv2d_condition.py'
lines = p.read_text().split('\n')

# Remove DIFFUSERS_CACHE, HF_HUB_OFFLINE, FLAX_WEIGHTS_NAME from the import block
# (lines 55, 56, 57 approximately)
filtered_lines = []
removed = set()
for i, line in enumerate(lines):
    stripped = line.strip()
    if stripped in ['DIFFUSERS_CACHE,', 'HF_HUB_OFFLINE,', 'FLAX_WEIGHTS_NAME,']:
        removed.add(stripped.rstrip(','))
        print(f'Removed line {i}: {repr(line)}')
        continue
    filtered_lines.append(line)

# Find where the big import block ends and add fallback assignments after it
# Look for the closing ')' of the from diffusers.utils import (...) block
in_block = False
insert_after = -1
for i, line in enumerate(filtered_lines):
    if 'from diffusers.utils import (' in line:
        in_block = True
    if in_block and line.strip() == ')':
        insert_after = i
        in_block = False
        break

if insert_after >= 0:
    fallbacks = [
        '# Fallback for removed constants',
        'try: from diffusers.utils import DIFFUSERS_CACHE',
        'except ImportError: DIFFUSERS_CACHE = os.path.expanduser("~/.cache/huggingface/diffusers")',
        'try: from diffusers.utils import HF_HUB_OFFLINE',
        'except ImportError: HF_HUB_OFFLINE = False',
        'try: from diffusers.utils import FLAX_WEIGHTS_NAME',
        'except ImportError: FLAX_WEIGHTS_NAME = "flax_model.msgpack"',
    ]
    for j, fb in enumerate(fallbacks):
        filtered_lines.insert(insert_after + 1 + j, fb)
    print(f'Added fallbacks after line {insert_after}')

# Also ensure _load_state_dict_into_model stub is there
# Check if stub exists
stub_exists = any('def _load_state_dict_into_model' in l for l in filtered_lines)
if not stub_exists:
    # Add stub after the try/except block for modeling_utils import
    for i, line in enumerate(filtered_lines):
        if 'from diffusers.models.modeling_utils import' in line and '_load_state_dict_into_model' not in line:
            # Add stub after this line
            stub_lines = [
                'try:',
                '    from diffusers.models.modeling_utils import _load_state_dict_into_model',
                'except ImportError:',
                "    def _load_state_dict_into_model(model, state_dict, start_prefix=''):",
                '        filtered = {k[len(start_prefix):] if k.startswith(start_prefix) else k: v for k, v in state_dict.items()}',
                '        model.load_state_dict(filtered, strict=False)',
                '        return [], []',
            ]
            for j, sl in enumerate(stub_lines):
                filtered_lines.insert(i + 1 + j, sl)
            print(f'Added _load_state_dict_into_model stub after line {i}')
            break
else:
    print('Stub already exists')

p.write_text('\n'.join(filtered_lines))
print('FIXED unet_mv2d_condition.py')

# ====== FIX pipeline: PIL is_directory issue ======
# The is_directory issue is from PIL._util which changed in newer PIL
# Fix it in the pipeline if it's imported there
pipe_file = root / 'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py'
if pipe_file.exists():
    sp = pipe_file.read_text()
    # Check what PIL import exists
    pil_lines = [l for l in sp.split('\n') if 'PIL' in l or 'from PIL' in l]
    print('Pipeline PIL imports:', pil_lines[:5])
    # The is_directory error comes from huggingface_hub internally.
    # It's likely triggered when importing from diffusers with newer PIL.
    # Not a direct code fix needed - it's a PIL compatibility issue in diffusers.

# ====== FIX save_cache: numpy.rec issue ======
# The numpy.rec issue is in trimesh or similar
util_file = root / 'utils/util.py'
if util_file.exists():
    su = util_file.read_text()
    # Check for numpy.rec usage
    for i, line in enumerate(su.split('\n')[:50]):
        if 'numpy' in line.lower() or 'rec' in line.lower():
            print(f'util.py line {i}: {repr(line[:100])}')

print('MASTER_FIX_DONE')

In [ ]:
import sys, os, importlib
# Clear any cached imports to force fresh reload
for mod_name in list(sys.modules.keys()):
    if 'mvdiffusion' in mod_name or 'automl' in mod_name or 'utils.util' in mod_name:
        del sys.modules[mod_name]

sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')

tests = [
    ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    ('MVDiffusionImagePipeline', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    ('AutoencoderKL', 'automl.AutoencoderKL', 'AutoencoderKL'),
    ('save_cache', 'utils.util', 'save_cache'),
]
n = 0
for label, mod, attr in tests:
    try:
        m2 = __import__(mod, fromlist=[attr])
        getattr(m2, attr)
        print('[PASS]', label)
        n += 1
    except Exception as e:
        print('[FAIL]', label, type(e).__name__, str(e)[:300])
print(f'Score: {n}/5')
print('SMOKE2_DONE')

In [ ]:
%pip install -q --force-reinstall "numpy==1.26.4" "pillow==9.5.0"
print('NUMPY_PIL_DOWNGRADED')

In [ ]:
from pathlib import Path
root = Path('/content/CADDreamer')

# Fix AutoencoderKL.py - check and fix IndentationError at line 20
vae_file = root / 'automl/AutoencoderKL.py'
vae_lines = vae_file.read_text().split('\n')

# Show lines 18-25 to see the issue
print("VAE lines 15-25:")
for i in range(15, min(26, len(vae_lines))):
    print(f"  {i}: {repr(vae_lines[i][:120])}")

# Fix: The FromOriginalVAEMixin try block might be empty
# Look for isolated try: blocks without proper indented body
for i, line in enumerate(vae_lines):
    if line.strip() == 'try:' and i + 1 < len(vae_lines):
        next_line = vae_lines[i + 1].strip()
        if next_line in ['', 'except ImportError:', 'except:'] or (not next_line.startswith((' ', '\t')) and next_line != ''):
            print(f'Found empty try: at line {i}, next line {i+1}: {repr(vae_lines[i+1])}')
            # Fix by adding pass
            vae_lines.insert(i + 1, '    pass')
            print(f'Added pass after line {i}')

vae_file.write_text('\n'.join(vae_lines))

# Verify
vae_lines2 = vae_file.read_text().split('\n')
print("\nFixed VAE lines 15-28:")
for i in range(15, min(29, len(vae_lines2))):
    print(f"  {i}: {repr(vae_lines2[i][:120])}")

print('VAE_FIX2_DONE')

In [ ]:
import subprocess, os
from pathlib import Path
root = Path('/content/CADDreamer')
os.chdir('/content/CADDreamer')

# Restore original AutoencoderKL.py from git
result = subprocess.run(['git', 'checkout', 'automl/AutoencoderKL.py'], capture_output=True, text=True, cwd=str(root))
print('git restore:', result.returncode, result.stderr[:200] if result.stderr else 'ok')

# Read the original
vae_file = root / 'automl/AutoencoderKL.py'
original = vae_file.read_text()

# Show the first 35 lines
lines = original.split('\n')
print("Original lines 0-35:")
for i in range(min(36, len(lines))):
    print(f"  {i}: {repr(lines[i][:120])}")
print('VAE_ORIG_DONE')

In [ ]:
from pathlib import Path
root = Path('/content/CADDreamer')

# ===== CLEAN PATCH AutoencoderKL.py =====
vae_file = root / 'automl/AutoencoderKL.py'
s = vae_file.read_text()

# Fix 1: FromOriginalVAEMixin
s = s.replace(
    'from diffusers.loaders import FromOriginalVAEMixin',
    'try:\n    from diffusers.loaders import FromOriginalVAEMixin\nexcept ImportError:\n    class FromOriginalVAEMixin: pass'
)

# Fix 2: apply_forward_hook
s = s.replace(
    'from diffusers.utils import BaseOutput, apply_forward_hook',
    'try:\n    from diffusers.utils import apply_forward_hook\nexcept ImportError:\n    def apply_forward_hook(fn): return fn\nfrom diffusers.utils import BaseOutput'
)

# Fix 3: diffusers.models.vae
s = s.replace(
    'from diffusers.models.vae import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder',
    'try:\n    from diffusers.models.vae import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder\nexcept ImportError:\n    pass  # local definitions will be used'
)

# Fix 4: diffusers.models.unet_2d_blocks
s = s.replace(
    'from diffusers.models.unet_2d_blocks import UNetMidBlock2D, get_down_block, get_up_block',
    'from diffusers.models.unets.unet_2d_blocks import UNetMidBlock2D, get_down_block, get_up_block'
)

# Fix 5: randn_tensor in diffusers.utils - already there in line 24
# Check if randn_tensor is importable from diffusers.utils in 0.29.2
# diffusers 0.29.2 has randn_tensor in diffusers.utils.torch_utils but also re-exported from diffusers.utils
# Let's just leave it as is (it should work with diffusers 0.29.2)

vae_file.write_text(s)

# Verify - show first 35 lines
lines = s.split('\n')
print("Patched AutoencoderKL.py lines 19-32:")
for i in range(19, min(33, len(lines))):
    print(f"  {i}: {repr(lines[i][:120])}")

# ===== ALSO FIX unet_mv2d_condition.py lines =====
# Need to also restore the original first to avoid accumulated patches
import subprocess
r = subprocess.run(['git', 'checkout', 'mvdiffusion/models/unet_mv2d_condition.py',
                   'mvdiffusion/models/unet_mv2d_blocks.py',
                   'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py',
                   'mvdiffusion/data/single_image_dataset.py'],
                   capture_output=True, text=True, cwd=str(root))
print('git restore mvdiffusion:', r.returncode)

# Now apply clean patches to unet_mv2d_condition.py
p = root / 'mvdiffusion/models/unet_mv2d_condition.py'
s2 = p.read_text()

# Fix _load_state_dict_into_model
s2 = s2.replace(
    'from diffusers.models.modeling_utils import ModelMixin, load_state_dict, _load_state_dict_into_model',
    'from diffusers.models.modeling_utils import ModelMixin, load_state_dict\ntry:\n    from diffusers.models.modeling_utils import _load_state_dict_into_model\nexcept ImportError:\n    def _load_state_dict_into_model(model, state_dict, start_prefix=""):\n        filtered = {k[len(start_prefix):] if k.startswith(start_prefix) else k: v for k, v in state_dict.items()}\n        model.load_state_dict(filtered, strict=False)\n        return [], []'
)

# Fix unet_2d_blocks import
s2 = s2.replace(
    'from diffusers.models.unet_2d_blocks import',
    'from diffusers.models.unets.unet_2d_blocks import'
)
# Fix dual_transformer_2d
s2 = s2.replace(
    'from diffusers.models.dual_transformer_2d import',
    'from diffusers.models.transformers.dual_transformer_2d import'
)
p.write_text(s2)

# Fix the multi-line DIFFUSERS_CACHE import block
lines2 = p.read_text().split('\n')
filtered = []
for line in lines2:
    if line.strip() in ['DIFFUSERS_CACHE,', 'HF_HUB_OFFLINE,', 'FLAX_WEIGHTS_NAME,']:
        continue
    filtered.append(line)

# Find closing ) of the import block and add fallbacks
in_block = False
insert_after = -1
for i, line in enumerate(filtered):
    if 'from diffusers.utils import (' in line:
        in_block = True
    if in_block and line.strip() == ')':
        insert_after = i
        in_block = False
        break

if insert_after >= 0:
    fallbacks = [
        'try: DIFFUSERS_CACHE = __import__("diffusers.utils", fromlist=["DIFFUSERS_CACHE"]).DIFFUSERS_CACHE',
        'except: DIFFUSERS_CACHE = __import__("os").path.expanduser("~/.cache/huggingface/diffusers")',
        'HF_HUB_OFFLINE = False',
        'FLAX_WEIGHTS_NAME = "flax_model.msgpack"',
    ]
    for j, fb in enumerate(fallbacks):
        filtered.insert(insert_after + 1 + j, fb)
    print(f'Added fallbacks after closing ) at line {insert_after}')

p.write_text('\n'.join(filtered))

# Fix unet_mv2d_blocks.py
p3 = root / 'mvdiffusion/models/unet_mv2d_blocks.py'
s3 = p3.read_text()
s3 = s3.replace('from diffusers.models.unet_2d_blocks import', 'from diffusers.models.unets.unet_2d_blocks import')
s3 = s3.replace('from diffusers.models.dual_transformer_2d import', 'from diffusers.models.transformers.dual_transformer_2d import')
s3 = s3.replace('from diffusers.models.attention import AdaGroupNorm',
                 'try:\n    from diffusers.models.attention import AdaGroupNorm\nexcept ImportError:\n    from diffusers.models.normalization import AdaGroupNorm')
p3.write_text(s3)

# Fix pipeline
p4 = root / 'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py'
s4 = p4.read_text()
s4 = s4.replace('from diffusers.utils import randn_tensor',
    'try:\n    from diffusers.utils import randn_tensor\nexcept ImportError:\n    from diffusers.utils.torch_utils import randn_tensor')
s4 = s4.replace('import logging\n', 'from diffusers.utils import logging as _diffusers_logging\n')
s4 = s4.replace('logging.getLogger(__name__)', '_diffusers_logging.get_logger(__name__)')
p4.write_text(s4)

# Fix single_image_dataset
p5 = root / 'mvdiffusion/data/single_image_dataset.py'
s5 = p5.read_text()
s5 = s5.replace('import cupy as cp', 'try:\n    import cupy as cp\nexcept ImportError:\n    cp = None')
p5.write_text(s5)

print('ALL_PATCHES_DONE')

In [ ]:
import sys, os

# Clear cached imports
for mod_name in list(sys.modules.keys()):
    if any(x in mod_name for x in ['mvdiffusion', 'automl', 'utils.util', 'diffusers', 'scipy', 'numpy', 'PIL']):
        del sys.modules[mod_name]

sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')

tests = [
    ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    ('MVDiffusionImagePipeline', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    ('AutoencoderKL', 'automl.AutoencoderKL', 'AutoencoderKL'),
    ('save_cache', 'utils.util', 'save_cache'),
]
n = 0
for label, mod, attr in tests:
    try:
        m2 = __import__(mod, fromlist=[attr])
        getattr(m2, attr)
        print('[PASS]', label)
        n += 1
    except Exception as e:
        print('[FAIL]', label, type(e).__name__, str(e)[:300])
print(f'Score: {n}/5')
print('SMOKE3_DONE')

In [ ]:
# Restore PIL to 11.3.0 (the binary was built for this) and numpy to 2.0.2
# Also upgrade scipy to be compatible with numpy 2.0
%pip install -q --force-reinstall "pillow==11.3.0"
%pip install -q --upgrade scipy
%pip install -q --force-reinstall "numpy==2.0.2"
print('ENV_RESTORED')

In [ ]:
# MASTER SETUP - Fresh runtime comprehensive setup
# Step A: Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('DRIVE MOUNTED')


In [ ]:
%%bash
# Step B: Install dependencies + restore repo from Drive
set +e

echo "=== Installing support deps ==="
pip install -q segment-anything trimesh meshio rembg onnxruntime scikit-image opencv-python dill 2>/dev/null

echo "=== Pinning core stack ==="
pip install -q --no-deps --force-reinstall --no-cache-dir   "diffusers==0.29.2"   "transformers==4.44.0"   "huggingface_hub==0.24.7"   "tokenizers==0.19.1"   "accelerate==0.33.0" 2>/dev/null

echo "=== Fixing scipy for numpy compat ==="
pip install -q --upgrade scipy 2>/dev/null

echo "=== Cloning CADDreamer repo ==="
if [ ! -d /content/CADDreamer ]; then
  git clone -q https://github.com/lidan233/CADDreamer /content/CADDreamer
fi
cd /content/CADDreamer

echo "=== Extracting Drive zips ==="
python3 - <<'PYEOF'
import zipfile, os, shutil
from pathlib import Path

drive_base = Path('/content/drive/MyDrive/caddreamer1')
root = Path('/content/CADDreamer')

# Extract ckpts
ckpt_zip = drive_base / 'ckpts.zip'
if ckpt_zip.exists():
    print(f'Extracting ckpts.zip ({ckpt_zip.stat().st_size//1024//1024}MB)...')
    with zipfile.ZipFile(ckpt_zip) as z:
        z.extractall(root)
    print('ckpts.zip done')
else:
    print('ERROR: ckpts.zip not found at', ckpt_zip)

# Extract finetuning_vae_normal
ft_zip = drive_base / 'finetuning_vae_normal.zip'
if ft_zip.exists():
    print(f'Extracting finetuning_vae_normal.zip ({ft_zip.stat().st_size//1024//1024}MB)...')
    with zipfile.ZipFile(ft_zip) as z:
        z.extractall(root)
    print('finetuning_vae_normal.zip done')
else:
    print('ERROR: finetuning_vae_normal.zip not found at', ft_zip)

# Extract image2csg_inputs
inp_zip = drive_base / 'image2csg_inputs.zip'
if inp_zip.exists():
    print(f'Extracting image2csg_inputs.zip ({inp_zip.stat().st_size//1024//1024}MB)...')
    with zipfile.ZipFile(inp_zip) as z:
        z.extractall(root)
    print('image2csg_inputs.zip done')
else:
    print('ERROR: image2csg_inputs.zip not found at', inp_zip)

# Verify key files
key_files = [
    'ckpts/zero123plus_v1.1.ckpt',
    'finetuning_vae_normal/pytorch_model.bin',
    'mvdiffusion/models/unet_mv2d_condition.py',
    'automl/AutoencoderKL.py',
]
for f in key_files:
    p = root / f
    exists = p.exists()
    size = p.stat().st_size if exists else 0
    print(f'  {"OK" if exists else "MISSING"}: {f} ({size} bytes)')

print('RESTORE_DONE')
PYEOF

echo "=== SETUP_PHASE_B_DONE ==="

In [ ]:
import os, subprocess

# Check the config file
result = subprocess.run(['cat', '/content/CADDreamer/configs/train/testing_4090_stage_1_cad_6views-lvis.yaml'],
                       capture_output=True, text=True)
print("CONFIG:", result.stdout[:3000])

# Check finetuning structure
result2 = subprocess.run(['find', '/content/CADDreamer/finetuning_vae_normal', '-type', 'f'],
                        capture_output=True, text=True)
print("FINETUNING FILES:", result2.stdout[:2000])

# Check ckpts/wonder3d-v1.0
result3 = subprocess.run(['ls', '/content/CADDreamer/ckpts/wonder3d-v1.0/'],
                        capture_output=True, text=True)
print("WONDER3D:", result3.stdout)

print("CONFIG_DIAG_DONE")

In [ ]:
from pathlib import Path
import os, sys

root = Path('/content/CADDreamer')

# ===== Fix 1: unet_mv2d_condition.py =====
p = root / 'mvdiffusion/models/unet_mv2d_condition.py'
s = p.read_text()

# Remove _load_state_dict_into_model from import if present
if '_load_state_dict_into_model' in s and 'def _load_state_dict_into_model' not in s:
    s = s.replace(', _load_state_dict_into_model', '')

# Fix unet_2d_blocks path
s = s.replace(
    'from diffusers.models.unet_2d_blocks import',
    'from diffusers.models.unets.unet_2d_blocks import'
)

# Remove DIFFUSERS_CACHE and HF_HUB_OFFLINE from imports
s = s.replace(' DIFFUSERS_CACHE,\n', '').replace(' HF_HUB_OFFLINE,\n', '')

# Add shims at top (idempotent)
shim = """
try:
    from diffusers.utils import DIFFUSERS_CACHE
except Exception:
    from pathlib import Path as _Path
    DIFFUSERS_CACHE = str(_Path.home() / ".cache/huggingface/diffusers")
try:
    from diffusers.utils import HF_HUB_OFFLINE
except Exception:
    HF_HUB_OFFLINE = False
try:
    from diffusers.models.modeling_utils import _load_state_dict_into_model
except Exception:
    def _load_state_dict_into_model(model_to_load, state_dict):
        model_to_load.load_state_dict(state_dict, strict=False)
        return []
"""
if 'DIFFUSERS_CACHE = str' not in s:
    # Find first import and insert shim after it
    lines = s.split('\n')
    insert_at = 0
    for i, line in enumerate(lines):
        if line.startswith('import ') or line.startswith('from '):
            insert_at = i + 1
            break
    lines.insert(insert_at, shim)
    s = '\n'.join(lines)

p.write_text(s)
print("Fix 1 unet_mv2d_condition.py: OK")

# ===== Fix 2: unet_mv2d_blocks.py =====
p = root / 'mvdiffusion/models/unet_mv2d_blocks.py'
s = p.read_text()
s = s.replace(
    'from diffusers.models.attention import AdaGroupNorm',
    'try:\n    from diffusers.models.attention import AdaGroupNorm\nexcept ImportError:\n    from diffusers.models.normalization import AdaGroupNorm'
)
s = s.replace(
    'from diffusers.models.dual_transformer_2d import',
    'from diffusers.models.transformers.dual_transformer_2d import'
)
s = s.replace(
    'from diffusers.models.unet_2d_blocks import',
    'from diffusers.models.unets.unet_2d_blocks import'
)
p.write_text(s)
print("Fix 2 unet_mv2d_blocks.py: OK")

# ===== Fix 3: transformer_mv2d.py - maybe_allow_in_graph =====
p = root / 'mvdiffusion/models/transformer_mv2d.py'
s = p.read_text()
if 'maybe_allow_in_graph' in s and 'def maybe_allow_in_graph' not in s:
    s = s.replace(
        'from diffusers.utils import BaseOutput, deprecate, maybe_allow_in_graph',
        """from diffusers.utils import BaseOutput, deprecate
try:
    from diffusers.utils.torch_utils import maybe_allow_in_graph
except ImportError:
    try:
        from diffusers.utils import maybe_allow_in_graph
    except ImportError:
        def maybe_allow_in_graph(fn):
            return fn"""
    )
p.write_text(s)
print("Fix 3 transformer_mv2d.py: OK")

# ===== Fix 4: pipeline_mvdiffusion_image.py - randn_tensor + logging =====
p = root / 'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py'
s = p.read_text()
# Fix randn_tensor
if 'from diffusers.utils import deprecate, logging, randn_tensor' in s:
    s = s.replace(
        'from diffusers.utils import deprecate, logging, randn_tensor',
        'from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor'
    )
elif 'randn_tensor' in s and 'from diffusers.utils.torch_utils import randn_tensor' not in s:
    # Add randn_tensor import
    s = s.replace(
        'from diffusers.utils import deprecate, logging',
        'from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor'
    )
p.write_text(s)
print("Fix 4 pipeline: OK")

# ===== Fix 5: AutoencoderKL.py =====
p = root / 'automl/AutoencoderKL.py'
s = p.read_text()

# Fix FromOriginalVAEMixin
if 'from diffusers.loaders import FromOriginalVAEMixin' in s and 'except' not in s.split('FromOriginalVAEMixin')[0][-100:]:
    s = s.replace(
        'from diffusers.loaders import FromOriginalVAEMixin',
        'try:\n    from diffusers.loaders import FromOriginalVAEMixin\nexcept Exception:\n    FromOriginalVAEMixin = object'
    )

# Fix apply_forward_hook
if 'from diffusers.utils import BaseOutput, apply_forward_hook' in s:
    s = s.replace(
        'from diffusers.utils import BaseOutput, apply_forward_hook',
        'from diffusers.utils import BaseOutput\ntry:\n    from diffusers.utils.accelerate_utils import apply_forward_hook\nexcept Exception:\n    apply_forward_hook = lambda fn: fn'
    )

# Fix vae import
if 'from diffusers.models.vae import' in s:
    s = s.replace(
        'from diffusers.models.vae import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder',
        'from diffusers.models.autoencoders.autoencoder_kl import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder'
    )

# Fix randn_tensor
if 'from diffusers.utils import BaseOutput, is_torch_version, randn_tensor' in s:
    s = s.replace(
        'from diffusers.utils import BaseOutput, is_torch_version, randn_tensor',
        'from diffusers.utils import BaseOutput, is_torch_version\nfrom diffusers.utils.torch_utils import randn_tensor'
    )
elif 'randn_tensor' in s and 'from diffusers.utils.torch_utils import randn_tensor' not in s:
    s = s.replace(
        'from diffusers.utils import BaseOutput, is_torch_version',
        'from diffusers.utils import BaseOutput, is_torch_version\nfrom diffusers.utils.torch_utils import randn_tensor'
    )

# Fix unet_2d_blocks
if 'from diffusers.models.unet_2d_blocks import' in s:
    s = s.replace(
        'from diffusers.models.unet_2d_blocks import',
        'from diffusers.models.unets.unet_2d_blocks import'
    )

p.write_text(s)
print("Fix 5 AutoencoderKL.py: OK")

# ===== Fix 6: single_image_dataset.py - cupy optional =====
p = root / 'mvdiffusion/data/single_image_dataset.py'
s = p.read_text()
if 'import cupy' in s and 'try:' not in s.split('import cupy')[0][-20:]:
    s = s.replace('import cupy', 'try:\n    import cupy\nexcept ImportError:\n    cupy = None')
p.write_text(s)
print("Fix 6 single_image_dataset.py: OK")

print("ALL_PATCHES_APPLIED")

In [ ]:
import sys, os, subprocess

# Run smoke test in a clean subprocess to avoid stale imports
test_script = """
import sys
import os
sys.path.insert(0, '/content/CADDreamer')
os.chdir('/content/CADDreamer')

tests = [
    ('UNetMV2DConditionModel', 'mvdiffusion.models.unet_mv2d_condition', 'UNetMV2DConditionModel'),
    ('SingleImageDataset', 'mvdiffusion.data.single_image_dataset', 'SingleImageDataset'),
    ('MVDiffusionImagePipeline', 'mvdiffusion.pipelines.pipeline_mvdiffusion_image', 'MVDiffusionImagePipeline'),
    ('AutoencoderKL', 'automl.AutoencoderKL', 'AutoencoderKL'),
    ('save_cache', 'utils.util', 'save_cache'),
]
n = 0
for label, mod, attr in tests:
    try:
        m2 = __import__(mod, fromlist=[attr])
        getattr(m2, attr)
        print('[PASS]', label)
        n += 1
    except Exception as e:
        print('[FAIL]', label, type(e).__name__, str(e)[:300])
print(f'Score: {n}/5')
print('SMOKE_DONE')
"""

result = subprocess.run(
    [sys.executable, '-c', test_script],
    capture_output=True, text=True,
    cwd='/content/CADDreamer',
    timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-2000:])


In [ ]:
import os, subprocess
from pathlib import Path

root = Path('/content/CADDreamer')
os.chdir(root)

# Check test images
for d in ['test_real_images', 'ckpts/testing_cases', 'image2csg_inputs']:
    p = root / d
    if p.exists():
        files = list(p.rglob('*.jpg')) + list(p.rglob('*.png'))
        print(f'{d}: {[str(f.relative_to(root)) for f in files[:5]]}')
    else:
        print(f'{d}: NOT FOUND')

# Ensure test_real_images dir has test.jpg
from PIL import Image
import numpy as np
test_dir = root / 'test_real_images'
test_dir.mkdir(exist_ok=True)
test_img = test_dir / 'test.jpg'
if not test_img.exists():
    img = Image.fromarray(np.random.randint(100, 200, (256,256,3), dtype=np.uint8))
    img.save(test_img)
    print(f'Created test.jpg at {test_img}')
else:
    print(f'test.jpg exists: {test_img.stat().st_size} bytes')

# Check if GPU available
import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print('INPUT_CHECK_DONE')


In [ ]:
from pathlib import Path
root = Path('/content/CADDreamer')
lines = (root/'test_mvdiffusion_seq.py').read_text().splitlines()
print('--- test_mvdiffusion_seq.py 405-455 ---')
for i in range(404, min(455, len(lines))):
    print(f'{i+1}: {lines[i]}')
print('--- dataset path logic ---')
dlines = (root/'mvdiffusion/data/single_image_dataset.py').read_text().splitlines()
keys = ('root_dir','filepath','filename','glob','listdir','Image.open')
seen=set()
for i,line in enumerate(dlines):
    if any(k in line for k in keys):
        for j in range(max(0,i-2), min(len(dlines),i+3)):
            if j not in seen:
                print(f'{j+1}: {dlines[j]}')
                seen.add(j)
print('PATH_LOGIC_DONE')

In [ ]:
import subprocess, os
from pathlib import Path
root = Path('/content/CADDreamer')
cmd = ['python3','-u','test_mvdiffusion_seq.py','--config','configs/train/testing_4090_stage_1_cad_6views-lvis.yaml','--idx','0','--gpu','0']
print('RUNNING', ' '.join(cmd), flush=True)
p = subprocess.Popen(cmd, cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
print('INFERENCE_RETURN_CODE', rc)
for base in ['test_outputs','test_outputs2','cached_output']:
    b = root/base
    if b.exists():
        files = [f for f in b.rglob('*') if f.is_file()]
        print('OUTPUT_DIR', base, 'FILES', len(files))
        for f in files[:80]:
            print(f.relative_to(root), f.stat().st_size)
print('INFERENCE_CELL_DONE')

In [ ]:
import subprocess
from pathlib import Path
root = Path('/content/CADDreamer')
u = subprocess.run(['python3','-m','pip','uninstall','-y','peft'], text=True, capture_output=True)
print(u.stdout)
print(u.stderr)
cmd = ['python3','-u','test_mvdiffusion_seq.py','--config','configs/train/testing_4090_stage_1_cad_6views-lvis.yaml','--idx','0','--gpu','0']
print('RETRYING', ' '.join(cmd), flush=True)
p = subprocess.Popen(cmd, cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
print('INFERENCE_RETURN_CODE', rc)
for base in ['test_outputs','test_outputs2']:
    b=root/base
    files=[f for f in b.rglob('*') if f.is_file()] if b.exists() else []
    print('OUTPUT_DIR',base,'FILES',len(files))
    for f in files[:100]:
        print(f.relative_to(root), f.stat().st_size)
print('RETRY_CELL_DONE')

In [ ]:
from pathlib import Path
import subprocess
root=Path('/content/CADDreamer')
script=root/'test_mvdiffusion_seq.py'
lines=script.read_text().splitlines()
patched=[]
disabled=0
for line in lines:
    if 'enable_xformers_memory_efficient_attention()' in line:
        indent=line[:len(line)-len(line.lstrip())]
        patched.append(indent+'# disabled on Colab T4: '+line.strip())
        disabled+=1
    else:
        patched.append(line)
script.write_text('\n'.join(patched)+'\n')
cfg=root/'configs/train/testing_4090_stage_1_cad_6views-lvis.yaml'
text=cfg.read_text().replace('enable_xformers_memory_efficient_attention: true','enable_xformers_memory_efficient_attention: false')
cfg.write_text(text)
print('XFORMERS_CALLS_DISABLED',disabled)
cmd=['python3','-u','test_mvdiffusion_seq.py','--config','configs/train/testing_4090_stage_1_cad_6views-lvis.yaml','--idx','0','--gpu','0']
print('RETRYING_NATIVE_ATTENTION',' '.join(cmd),flush=True)
p=subprocess.Popen(cmd,cwd=root,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in p.stdout:
    print(line,end='',flush=True)
rc=p.wait()
print('INFERENCE_RETURN_CODE',rc)
for base in ['test_outputs','test_outputs2']:
    b=root/base
    files=[f for f in b.rglob('*') if f.is_file()] if b.exists() else []
    print('OUTPUT_DIR',base,'FILES',len(files))
    for f in files[:120]:
        print(f.relative_to(root),f.stat().st_size)
print('NATIVE_RETRY_DONE')

In [ ]:
from pathlib import Path
root=Path('/content/CADDreamer')
for p in list(root.rglob('requirements*.txt'))+list(root.rglob('environment*.yml'))+list(root.rglob('pyproject.toml')):
    try:
        text=p.read_text(errors='ignore')
    except Exception:
        continue
    hits=[line for line in text.splitlines() if any(k in line.lower() for k in ('distloss','tinycudann','nerfacc','pytorch-lightning','torchmetrics'))]
    if hits:
        print('FILE',p.relative_to(root))
        for h in hits: print(h)
print('DEPENDENCY_SEARCH_DONE')

In [ ]:
import subprocess
from pathlib import Path
root=Path('/content/CADDreamer')
install=subprocess.run(['python3','-m','pip','install','-q','torch_efficient_distloss==0.1.3'],text=True,capture_output=True)
print('INSTALL_RC',install.returncode)
print(install.stdout)
print(install.stderr)
runner = """
from instant_nsr_pl.launch import main as neus_main
from pathlib import Path
root=Path('/content/CADDreamer')
input_root=root/'test_outputs'
scene=input_root/'cropsize-256-cfg1.0/0_0deepcad'
merge_cfg={'gpu':'0','config':str(root/'neus/configs/neuralangelo-ortho-wmask.yaml'),'dataset':['dataset.root_dir='+str(input_root),'dataset.scene='+str(scene)]}
print('NEUS_CONFIG',merge_cfg,flush=True)
neus_main(merge_cfg)
print('NEUS_MAIN_DONE',flush=True)
"""
p=subprocess.Popen(['python3','-u','-c',runner],cwd=root,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in p.stdout:
    print(line,end='',flush=True)
rc=p.wait()
print('NEUS_RETURN_CODE',rc)
scene=root/'test_outputs/cropsize-256-cfg1.0/0_0deepcad'
for f in sorted([p for p in scene.rglob('*') if p.is_file()]):
    if f.suffix.lower() in ('.obj','.ply','.stl','.step','.png','.ckpt'):
        print('RESULT',f.relative_to(root),f.stat().st_size)
print('MESH_CELL_DONE')

In [ ]:
from pathlib import Path
print(Path('/content/CADDreamer/requirements.txt').read_text())
print('REQUIREMENTS_DONE')

In [ ]:
import subprocess
from pathlib import Path
root=Path('/content/CADDreamer')
install=subprocess.run(['python3','-m','pip','install','-q','pytorch-lightning==1.9.5','torchmetrics==1.4.1'],text=True,capture_output=True)
print('INSTALL_RC',install.returncode)
print(install.stdout)
print(install.stderr)
runner="""
from instant_nsr_pl.launch import main as neus_main
from pathlib import Path
root=Path('/content/CADDreamer')
input_root=root/'test_outputs'
scene=input_root/'cropsize-256-cfg1.0/0_0deepcad'
merge_cfg={'gpu':'0','config':str(root/'neus/configs/neuralangelo-ortho-wmask.yaml'),'dataset':['dataset.root_dir='+str(input_root),'dataset.scene='+str(scene)]}
print('NEUS_CONFIG',merge_cfg,flush=True)
neus_main(merge_cfg)
print('NEUS_MAIN_DONE',flush=True)
"""
p=subprocess.Popen(['python3','-u','-c',runner],cwd=root,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in p.stdout:
    print(line,end='',flush=True)
rc=p.wait()
print('NEUS_RETURN_CODE',rc)
scene=root/'test_outputs/cropsize-256-cfg1.0/0_0deepcad'
for f in sorted([p for p in scene.rglob('*') if p.is_file()]):
    if f.suffix.lower() in ('.obj','.ply','.stl','.step','.ckpt'):
        print('MESH_RESULT',f.relative_to(root),f.stat().st_size)
print('MESH_RETRY_DONE')

In [ ]:
from pathlib import Path
root=Path('/content/CADDreamer')
for p in root.rglob('*'):
    if p.is_file() and p.suffix.lower() in ('.md','.txt','.sh','.yaml','.yml','.toml'):
        try: text=p.read_text(errors='ignore')
        except: continue
        if 'tinycudann' in text.lower() or 'tiny-cuda-nn' in text.lower():
            print('FILE',p.relative_to(root))
            for n,line in enumerate(text.splitlines(),1):
                if 'tinycudann' in line.lower() or 'tiny-cuda-nn' in line.lower():
                    print(n,line)
print('TCNN_SEARCH_DONE')

In [ ]:
import subprocess, os
from pathlib import Path
root=Path('/content/CADDreamer')
env=os.environ.copy()
env['TCNN_CUDA_ARCHITECTURES']='75'
cmd_install=['python3','-m','pip','install','-v','git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch']
print('INSTALLING_TCNN',' '.join(cmd_install),flush=True)
p=subprocess.Popen(cmd_install,cwd=root,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in p.stdout:
    print(line,end='',flush=True)
install_rc=p.wait()
print('TCNN_INSTALL_RETURN_CODE',install_rc,flush=True)
if install_rc==0:
    runner="""
from instant_nsr_pl.launch import main as neus_main
from pathlib import Path
root=Path('/content/CADDreamer')
input_root=root/'test_outputs'
scene=input_root/'cropsize-256-cfg1.0/0_0deepcad'
merge_cfg={'gpu':'0','config':str(root/'neus/configs/neuralangelo-ortho-wmask.yaml'),'dataset':['dataset.root_dir='+str(input_root),'dataset.scene='+str(scene)]}
print('NEUS_CONFIG',merge_cfg,flush=True)
neus_main(merge_cfg)
print('NEUS_MAIN_DONE',flush=True)
"""
    q=subprocess.Popen(['python3','-u','-c',runner],cwd=root,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in q.stdout:
        print(line,end='',flush=True)
    rc=q.wait()
    print('NEUS_RETURN_CODE',rc)
print('TCNN_MESH_CELL_DONE')

In [ ]:
import subprocess, os
from pathlib import Path
root = Path('/content/CADDreamer')
cmds = [
    ['python3', '-m', 'pip', 'install', '-q', 'ninja', 'torch_efficient_distloss==0.1.3', 'pytorch-lightning==1.9.5', 'torchmetrics==1.4.1', 'nerfacc'],
    ['python3', '-m', 'pip', 'install', '-v', 'git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch'],
]
env = os.environ.copy()
env['TCNN_CUDA_ARCHITECTURES'] = '75'
for cmd in cmds:
    print('RUN', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=root, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    print('CMD_RETURN_CODE', rc, flush=True)
    if rc != 0:
        raise SystemExit(rc)
runner = """
from instant_nsr_pl.launch import main as neus_main
from pathlib import Path
root = Path('/content/CADDreamer')
input_root = root / 'test_outputs'
scene = input_root / 'cropsize-256-cfg1.0/0_0deepcad'
merge_cfg = {
    'gpu': '0',
    'config': str(root / 'neus/configs/neuralangelo-ortho-wmask.yaml'),
    'dataset': [
        'dataset.root_dir=' + str(input_root),
        'dataset.scene=' + str(scene),
    ],
}
print('NEUS_CONFIG', merge_cfg, flush=True)
neus_main(merge_cfg)
print('NEUS_MAIN_DONE', flush=True)
"""
q = subprocess.Popen(['python3', '-u', '-c', runner], cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in q.stdout:
    print(line, end='', flush=True)
rc = q.wait()
print('NEUS_RETURN_CODE', rc, flush=True)

In [ ]:
%%bash
# COMPLETE SETUP: Install + Clone + Extract + Patch (T4 GPU)
set +e

echo "=== Step 1: Install dependencies ==="
pip install -q segment-anything trimesh meshio rembg onnxruntime scikit-image opencv-python dill 2>/dev/null
pip install -q --no-deps --force-reinstall --no-cache-dir "diffusers==0.29.2" "transformers==4.44.0" "huggingface_hub==0.24.7" "tokenizers==0.19.1" "accelerate==0.33.0" 2>/dev/null
pip install -q --upgrade scipy 2>/dev/null
echo "deps done"

echo "=== Step 2: Clone CADDreamer ==="
if [ ! -d /content/CADDreamer ]; then
  git clone -q https://github.com/lidan233/CADDreamer /content/CADDreamer
fi
echo "repo ready"

echo "=== Step 3: Extract from Drive ==="
python3 << 'PYEOF'
import zipfile
from pathlib import Path
drive = Path('/content/drive/MyDrive/caddreamer1')
root = Path('/content/CADDreamer')
for zname in ['ckpts.zip', 'finetuning_vae_normal.zip', 'image2csg_inputs.zip']:
    z = drive / zname
    if z.exists():
        print(f'Extracting {zname} ({z.stat().st_size//1024//1024}MB)...')
        with zipfile.ZipFile(z) as zf:
            zf.extractall(root)
        print(f'{zname} done')
    else:
        print(f'WARNING: {zname} not found')
print('EXTRACT_DONE')
PYEOF

echo "=== Step 4: Apply patches ==="
python3 << 'PYEOF'
from pathlib import Path
root = Path('/content/CADDreamer')

# Fix unet_mv2d_condition.py
p = root/'mvdiffusion/models/unet_mv2d_condition.py'; s = p.read_text()
s = s.replace(', _load_state_dict_into_model','') if 'def _load_state_dict_into_model' not in s else s
s = s.replace('from diffusers.models.unet_2d_blocks import','from diffusers.models.unets.unet_2d_blocks import')
s = s.replace(' DIFFUSERS_CACHE,
','').replace(' HF_HUB_OFFLINE,
','')
shim='
try:
    from diffusers.utils import DIFFUSERS_CACHE
except:
    DIFFUSERS_CACHE=str(Path.home()/".cache/huggingface/diffusers")
try:
    from diffusers.utils import HF_HUB_OFFLINE
except:
    HF_HUB_OFFLINE=False
try:
    from diffusers.models.modeling_utils import _load_state_dict_into_model
except:
    def _load_state_dict_into_model(m,sd):
        m.load_state_dict(sd,strict=False); return []
'
if 'DIFFUSERS_CACHE = str' not in s: s='from pathlib import Path
'+shim+s
p.write_text(s); print("unet_mv2d_condition.py OK")

# Fix unet_mv2d_blocks.py
p = root/'mvdiffusion/models/unet_mv2d_blocks.py'; s = p.read_text()
s = s.replace('from diffusers.models.attention import AdaGroupNorm','try:
    from diffusers.models.attention import AdaGroupNorm
except ImportError:
    from diffusers.models.normalization import AdaGroupNorm')
s = s.replace('from diffusers.models.dual_transformer_2d import','from diffusers.models.transformers.dual_transformer_2d import')
s = s.replace('from diffusers.models.unet_2d_blocks import','from diffusers.models.unets.unet_2d_blocks import')
p.write_text(s); print("unet_mv2d_blocks.py OK")

# Fix transformer_mv2d.py
p = root/'mvdiffusion/models/transformer_mv2d.py'; s = p.read_text()
if 'maybe_allow_in_graph' in s and 'def maybe_allow_in_graph' not in s:
    s = s.replace('from diffusers.utils import BaseOutput, deprecate, maybe_allow_in_graph','from diffusers.utils import BaseOutput, deprecate
try:
    from diffusers.utils.torch_utils import maybe_allow_in_graph
except ImportError:
    def maybe_allow_in_graph(fn): return fn')
p.write_text(s); print("transformer_mv2d.py OK")

# Fix pipeline
p = root/'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py'; s = p.read_text()
if 'from diffusers.utils.torch_utils import randn_tensor' not in s:
    s = s.replace('from diffusers.utils import deprecate, logging, randn_tensor','from diffusers.utils import deprecate, logging
from diffusers.utils.torch_utils import randn_tensor')
    if 'from diffusers.utils.torch_utils import randn_tensor' not in s:
        s = s.replace('from diffusers.utils import deprecate, logging','from diffusers.utils import deprecate, logging
from diffusers.utils.torch_utils import randn_tensor')
p.write_text(s); print("pipeline OK")

# Fix AutoencoderKL.py
p = root/'automl/AutoencoderKL.py'; s = p.read_text()
s = s.replace('from diffusers.loaders import FromOriginalVAEMixin','try:
    from diffusers.loaders import FromOriginalVAEMixin
except:
    FromOriginalVAEMixin=object')
s = s.replace('from diffusers.utils import BaseOutput, apply_forward_hook','from diffusers.utils import BaseOutput
try:
    from diffusers.utils.accelerate_utils import apply_forward_hook
except:
    apply_forward_hook=lambda fn:fn')
s = s.replace('from diffusers.models.vae import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder','from diffusers.models.autoencoders.autoencoder_kl import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder')
if 'from diffusers.utils.torch_utils import randn_tensor' not in s:
    s = s.replace('from diffusers.utils import BaseOutput, is_torch_version, randn_tensor','from diffusers.utils import BaseOutput, is_torch_version
from diffusers.utils.torch_utils import randn_tensor')
    if 'from diffusers.utils.torch_utils import randn_tensor' not in s:
        s = s.replace('from diffusers.utils import BaseOutput, is_torch_version','from diffusers.utils import BaseOutput, is_torch_version
from diffusers.utils.torch_utils import randn_tensor')
s = s.replace('from diffusers.models.unet_2d_blocks import','from diffusers.models.unets.unet_2d_blocks import')
p.write_text(s); print("AutoencoderKL.py OK")

# Fix single_image_dataset.py
p = root/'mvdiffusion/data/single_image_dataset.py'; s = p.read_text()
if 'import cupy' in s and 'try:' not in s.split('import cupy')[0][-20:]:
    s = s.replace('import cupy','try:
    import cupy
except ImportError:
    cupy=None')
p.write_text(s); print("single_image_dataset.py OK")
print("ALL_PATCHES_DONE")
PYEOF

echo "=== SETUP COMPLETE ==="

In [ ]:
import subprocess
r = subprocess.run(['ls', '/content/CADDreamer/'], capture_output=True, text=True)
print("CADDreamer exists:", r.returncode == 0)
print(r.stdout[:200])
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('DRIVE OK')

Mounted at /content/drive
DRIVE OK


In [ ]:
import os, sys, shutil, zipfile, subprocess
from pathlib import Path

ROOT = Path('/content/CADDreamer')
DRIVE = Path('/content/drive/MyDrive/caddreamer1')

def run(cmd, cwd=None, env=None, check=True):
    print('RUN:', ' '.join(map(str, cmd)), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    print('RETURN_CODE:', rc, flush=True)
    if check and rc:
        raise RuntimeError('command failed: ' + ' '.join(map(str, cmd)))
    return rc

print('=== Restore repository and assets ===', flush=True)
assert DRIVE.exists(), f'Drive folder missing: {DRIVE}'
if not ROOT.exists():
    run(['git', 'clone', 'https://github.com/lidan233/CADDreamer.git', str(ROOT)], cwd='/content')

for name in ['ckpts.zip', 'finetuning_vae_normal.zip', 'image2csg_inputs.zip']:
    src = DRIVE / name
    dst = Path('/content') / name
    assert src.exists(), f'Missing archive: {src}'
    if (not dst.exists()) or dst.stat().st_size != src.stat().st_size:
        print('COPY:', src, '->', dst, flush=True)
        shutil.copyfile(src, dst)
    print('EXTRACT:', dst, flush=True)
    with zipfile.ZipFile(dst) as zf:
        zf.extractall(ROOT)

print('=== Install compatible Python stack ===', flush=True)
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
     'ninja', 'einops', 'omegaconf', 'rembg', 'onnxruntime', 'segment-anything',
     'trimesh', 'meshio', 'open3d', 'scikit-image', 'opencv-python', 'dill',
     'torch_efficient_distloss==0.1.3', 'pytorch-lightning==1.9.5',
     'torchmetrics==1.4.1', 'nerfacc==0.3.5'])
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', '--no-cache-dir',
     'huggingface_hub==0.24.7', 'tokenizers==0.19.1', 'transformers==4.44.0',
     'accelerate==0.33.0', 'diffusers==0.29.2', 'numpy==2.4.2', 'pillow==9.5.0'])
run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'], check=False)

print('=== Place assets at paths expected by scripts ===', flush=True)
copies = [
    (ROOT/'image2csg_inputs/test_real_images/testnormal_0_0.png', ROOT/'our_inputs/test_real_images/testnormal_0_0.png'),
    (ROOT/'finetuning_vae_normal/outputs/finetune_vae/vae_16.pth', ROOT/'finetuning_vae_normal/outputs/vae_16.pth'),
    (ROOT/'finetuning_vae_normal/outputs/finetune_vae/mlp_16.pth', ROOT/'finetuning_vae_normal/outputs/mlp_16.pth'),
    (ROOT/'finetuning_vae_normal/outputs/finetuning_vae_normal/vae_normal_2.pth', ROOT/'finetuning_vae_normal/outputs/vae_normal_2.pth'),
]
for src, dst in copies:
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print('PLACED:', dst, flush=True)
    else:
        print('SOURCE_MISSING:', src, flush=True)

print('=== Apply Colab compatibility patches ===', flush=True)
p = ROOT/'mvdiffusion/models/unet_mv2d_condition.py'
s = p.read_text()
s = s.replace('from diffusers.models.modeling_utils import ModelMixin, load_state_dict, _load_state_dict_into_model',
              'from diffusers.models.modeling_utils import ModelMixin, load_state_dict')
s = s.replace('from diffusers.models.unet_2d_blocks import',
              'from diffusers.models.unets.unet_2d_blocks import')
s = s.replace('    DIFFUSERS_CACHE,\n', '').replace('    HF_HUB_OFFLINE,\n', '')
shim = """try:
    from diffusers.utils import DIFFUSERS_CACHE
except Exception:
    DIFFUSERS_CACHE = os.path.expanduser('~/.cache/huggingface/diffusers')
try:
    from diffusers.utils import HF_HUB_OFFLINE
except Exception:
    HF_HUB_OFFLINE = False
try:
    from diffusers.models.modeling_utils import _load_state_dict_into_model
except Exception:
    def _load_state_dict_into_model(model_to_load, state_dict):
        model_to_load.load_state_dict(state_dict, strict=False)
        return []
"""
if 'DIFFUSERS_CACHE = os.path.expanduser' not in s:
    s = 'import os\n' + shim + s
p.write_text(s)

p = ROOT/'mvdiffusion/models/unet_mv2d_blocks.py'
s = p.read_text()
s = s.replace('from diffusers.models.attention import AdaGroupNorm',
              'from diffusers.models.normalization import AdaGroupNorm')
s = s.replace('from diffusers.models.dual_transformer_2d import',
              'from diffusers.models.transformers.dual_transformer_2d import')
s = s.replace('from diffusers.models.unet_2d_blocks import',
              'from diffusers.models.unets.unet_2d_blocks import')
p.write_text(s)

p = ROOT/'mvdiffusion/models/transformer_mv2d.py'
s = p.read_text()
s = s.replace('from diffusers.utils import BaseOutput, deprecate, maybe_allow_in_graph',
"""from diffusers.utils import BaseOutput, deprecate
try:
    from diffusers.utils.torch_utils import maybe_allow_in_graph
except ImportError:
    def maybe_allow_in_graph(fn):
        return fn""")
p.write_text(s)

p = ROOT/'mvdiffusion/pipelines/pipeline_mvdiffusion_image.py'
s = p.read_text()
s = s.replace('from diffusers.utils import deprecate, logging, randn_tensor',
              'from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor')
s = s.replace('from diffusers.utils import deprecate, logging as dlogging, randn_tensor',
              'from diffusers.utils import deprecate, logging\nfrom diffusers.utils.torch_utils import randn_tensor')
s = s.replace('dlogging.get_logger', 'logging.get_logger')
p.write_text(s)

p = ROOT/'automl/AutoencoderKL.py'
s = p.read_text()
s = s.replace('from diffusers.loaders import FromOriginalVAEMixin',
              'try:\n from diffusers.loaders import FromOriginalVAEMixin\nexcept Exception:\n FromOriginalVAEMixin = object')
s = s.replace('from diffusers.utils import BaseOutput, apply_forward_hook',
              'from diffusers.utils import BaseOutput\ntry:\n from diffusers.utils.accelerate_utils import apply_forward_hook\nexcept Exception:\n apply_forward_hook = lambda fn: fn')
s = s.replace('from diffusers.models.vae import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder',
              'from diffusers.models.autoencoders.autoencoder_kl import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder')
s = s.replace('from diffusers.utils import BaseOutput, is_torch_version, randn_tensor',
              'from diffusers.utils import BaseOutput, is_torch_version\nfrom diffusers.utils.torch_utils import randn_tensor')
s = s.replace('from diffusers.models.unet_2d_blocks import',
              'from diffusers.models.unets.unet_2d_blocks import')
p.write_text(s)

p = ROOT/'mvdiffusion/data/single_image_dataset.py'
s = p.read_text()
if 'import cupy' in s and 'try:\n    import cupy' not in s:
    s = s.replace('import cupy', 'try:\n    import cupy\nexcept ImportError:\n    cupy = None')
p.write_text(s)

p = ROOT/'test_mvdiffusion_seq.py'
s = p.read_text()
s = s.replace('if cfg.enable_xformers_memory_efficient_attention:',
              'if False and cfg.enable_xformers_memory_efficient_attention:')
s = s.replace('pipeline.unet.enable_xformers_memory_efficient_attention()',
              "print('skip xformers; using native attention')")
p.write_text(s)

print('=== Verify restored assets and imports ===', flush=True)
checks = [
    ROOT/'ckpts/wonder3d-v1.0/model_index.json',
    ROOT/'ckpts/wonder3d-v1.0/unet/diffusion_pytorch_model.bin',
    ROOT/'finetuning_vae_normal/outputs/vae_16.pth',
    ROOT/'finetuning_vae_normal/outputs/mlp_16.pth',
    ROOT/'finetuning_vae_normal/outputs/vae_normal_2.pth',
    ROOT/'our_inputs/test_real_images/testnormal_0_0.png',
]
for item in checks:
    print(('OK' if item.exists() else 'MISSING'), item, flush=True)
assert all(item.exists() for item in checks), 'Required restored assets are missing'

test = """import os,sys
sys.path.insert(0,'/content/CADDreamer')
os.chdir('/content/CADDreamer')
from mvdiffusion.models.unet_mv2d_condition import UNetMV2DConditionModel
from mvdiffusion.data.single_image_dataset import SingleImageDataset
from mvdiffusion.pipelines.pipeline_mvdiffusion_image import MVDiffusionImagePipeline
from automl.AutoencoderKL import AutoencoderKL
from nerfacc import ContractionType, OccupancyGrid
print('CORE_IMPORTS_PASS')
print('NERFACC_COMPAT_PASS')
"""
run([sys.executable, '-u', '-c', test], cwd=ROOT)
print('RESTORE_AND_COMPATIBILITY_DONE', flush=True)

=== Restore repository and assets ===
RUN: git clone https://github.com/lidan233/CADDreamer.git /content/CADDreamer
Cloning into '/content/CADDreamer'...
RETURN_CODE: 0
COPY: /content/drive/MyDrive/caddreamer1/ckpts.zip -> /content/ckpts.zip
EXTRACT: /content/ckpts.zip
COPY: /content/drive/MyDrive/caddreamer1/finetuning_vae_normal.zip -> /content/finetuning_vae_normal.zip
EXTRACT: /content/finetuning_vae_normal.zip
COPY: /content/drive/MyDrive/caddreamer1/image2csg_inputs.zip -> /content/image2csg_inputs.zip
EXTRACT: /content/image2csg_inputs.zip
=== Install compatible Python stack ===
RUN: /usr/bin/python3 -m pip install -q --no-cache-dir ninja einops omegaconf rembg onnxruntime segment-anything trimesh meshio open3d scikit-image opencv-python dill torch_efficient_distloss==0.1.3 pytorch-lightning==1.9.5 torchmetrics==1.4.1 nerfacc==0.3.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 150.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.2/866.2 kB 241.4 MB/

In [ ]:
from pathlib import Path
import subprocess, sys

p = Path('/content/CADDreamer/automl/AutoencoderKL.py')
s = p.read_text()
start = s.index('from dataclasses import dataclass')
marker = s.index('class Decoder')
prefix = s[:start]
rest = s[marker:]
imports = """from dataclasses import dataclass
from typing import Dict, Optional, Tuple, Union

import torch
import torch.nn as nn
from diffusers.configuration_utils import ConfigMixin, register_to_config
try:
    from diffusers.loaders import FromOriginalVAEMixin
except Exception:
    FromOriginalVAEMixin = object
from diffusers.utils import BaseOutput
try:
    from diffusers.utils.accelerate_utils import apply_forward_hook
except Exception:
    def apply_forward_hook(fn):
        return fn
from diffusers.models.attention_processor import AttentionProcessor, AttnProcessor, SpatialNorm
from diffusers.models.modeling_utils import ModelMixin
from diffusers.models.autoencoders.autoencoder_kl import Decoder, DecoderOutput, DiagonalGaussianDistribution, Encoder
from diffusers.utils import is_torch_version
from diffusers.utils.torch_utils import randn_tensor
from diffusers.models.unets.unet_2d_blocks import UNetMidBlock2D, get_down_block, get_up_block


"""
p.write_text(prefix + imports + rest)
print('AUTOENCODER_HEADER_REPAIRED')

test = """import os,sys,importlib.metadata
sys.path.insert(0,'/content/CADDreamer')
os.chdir('/content/CADDreamer')
from mvdiffusion.models.unet_mv2d_condition import UNetMV2DConditionModel
from mvdiffusion.data.single_image_dataset import SingleImageDataset
from mvdiffusion.pipelines.pipeline_mvdiffusion_image import MVDiffusionImagePipeline
from automl.AutoencoderKL import AutoencoderKL
from nerfacc import ContractionType, OccupancyGrid, ray_marching, render_weight_from_density, accumulate_along_rays
print('nerfacc', importlib.metadata.version('nerfacc'))
print('CORE_IMPORTS_PASS')
print('NERFACC_COMPAT_PASS')
"""
r = subprocess.run([sys.executable, '-u', '-c', test], cwd='/content/CADDreamer', text=True, capture_output=True)
print(r.stdout)
print(r.stderr)
print('IMPORT_TEST_RETURN_CODE', r.returncode)
assert r.returncode == 0
print('RESTORE_AND_COMPATIBILITY_DONE')

In [ ]:
import subprocess, os
from pathlib import Path

root = Path('/content/CADDreamer')
cmd = [
    'python3', '-u', 'test_mvdiffusion_seq.py',
    '--config', 'configs/train/testing_4090_stage_1_cad_6views-lvis.yaml',
    '--idx', '0'
]
print('STAGE1_RUN', ' '.join(cmd), flush=True)
p = subprocess.Popen(cmd, cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
print('STAGE1_RETURN_CODE', rc, flush=True)

scene = root/'test_outputs/cropsize-256-cfg1.0/0_0deepcad'
print('SCENE_EXISTS', scene.exists(), flush=True)
if scene.exists():
    for f in sorted(scene.rglob('*')):
        if f.is_file():
            print('OUTPUT', f.relative_to(root), f.stat().st_size, flush=True)
print('STAGE1_CELL_DONE', flush=True)

STAGE1_RUN python3 -u test_mvdiffusion_seq.py --config configs/train/testing_4090_stage_1_cad_6views-lvis.yaml --idx 0
2026-06-24 06:09:33.166896: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
{'pretrained_model_name_or_path': './ckpts/wonder3d-v1.0', 'revision': None, 'validation_dataset': {'root_dir': './test_real_images', 'num_views': 6, 'bg_color': 'white', 'img_wh': [256, 256], 'crop_size': 256, 'filepath': 'test.jpg'}, 'save_dir': './test_outputs', 'seed': 42, 'validation_batch_size': 1, 'dataloader_num_workers': 0, 'local_rank': -1, 'pipe_kwargs': {'camera_embedding_type': 'e_de_da_sincos', 'num_views': 6}, 'pipe_validation_kwargs': {'eta': 1.0}, 'unet_from_pretrained_kwargs': {'camera_embedding_type': 'e_de_da_sincos', 'projection_class

In [ ]:
import shutil
from pathlib import Path

src = Path("/content/CADDreamer/test_outputs")
dst = Path("/content/drive/MyDrive/caddreamer1/runtime_backup/test_outputs")
shutil.copytree(src, dst, dirs_exist_ok=True)
print("BACKUP_DONE", dst)

BACKUP_DONE /content/drive/MyDrive/caddreamer1/runtime_backup/test_outputs
